In [4]:
def apply_sparse_cca(X, Z, X_cols=None, Z_cols=None, 
                     standardize=True, 
                     find_best_n=True,
                     find_best_penalty=True,
                     penalty_range=None,
                     n_components=None,
                     penaltyu=None,
                     penaltyv=None,
                     n_splits=5,
                     random_state=42,
                     do_permutation_test=True,
                     n_permutations=1000,
                     visualize=True,
                     verbose=True):
    
    # 데이터 준비
    if isinstance(X, pd.DataFrame):
        X = X.values
    if isinstance(Z, pd.DataFrame):
        Z = Z.values
    
    # 표준화
    if standardize:
        X = ((X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12))
        Z = ((Z - Z.mean(axis=0)) / (Z.std(axis=0) + 1e-12))
    
    # 1. 최적 컴포넌트 수 결정
    if find_best_n:
        if verbose:
            print("=" * 50)
            print("Finding optimal number of components...")
            print("=" * 50)

        max_k = min(2, X.shape[1], Z.shape[1], X.shape[0] - 1)

        if max_k < 1:
            raise ValueError("Not enough samples/features to run CCA (max_k < 1).")

        # ✅ Z가 1개(또는 min이 1)면 탐색 생략하고 1로 고정
        if max_k == 1:
            best_n = 1
            if verbose:
                print("max_k==1 -> set n_components(best_n)=1 (skip search)")
        else:
            cca = CCA(n_components=max_k)
            U_cca, V_cca = cca.fit_transform(X, Z)

            cors = []
            for i in range(max_k):
                r = np.corrcoef(U_cca[:, i], V_cca[:, i])[0, 1]
                cors.append(r)
                if verbose:
                    print(f"Component {i+1}: correlation = {r:.4f}")

            # 급격히 떨어지는 지점 찾기
            best_n = 1
            for i in range(len(cors) - 1):
                if cors[i] - cors[i + 1] > 0.1:
                    best_n = i + 1
                    break
            else:
                best_n = len([c for c in cors if c > 0.1])

            best_n = max(1, min(best_n, max_k))

            if verbose:
                print(f"\nRecommended n_components: {best_n}\n") 
    else:
        if n_components is None:
            raise ValueError("n_components must be specified when find_best_n=False")
        best_n = n_components
    
    # 2. 최적 penalty 파라미터 찾기
    if find_best_penalty:
        if verbose:
            print("=" * 50)
            print("Grid search for best penalty parameters...")
            print("=" * 50)
        
        if penalty_range is None:
            penalty_range = np.arange(0.1, 1.0, 0.1)
        
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        best_score = -np.inf
        best_penaltyu = None
        best_penaltyv = None
        results_grid = []
        
        for penaltyu, penaltyv in itertools.product(penalty_range, penalty_range):
            cv_scores = []
            
            for train_idx, val_idx in kf.split(X):
                X_train, X_val = X[train_idx], X[val_idx]
                Z_train, Z_val = Z[train_idx], Z[val_idx]
                
                try:
                    U, V, D = pmd(X_train.T @ Z_train, K=best_n,
                                 penaltyu=penaltyu, penaltyv=penaltyv,
                                 standardize=False)
                    
                    X_scores = X_val @ U[:, :best_n]
                    Z_scores = Z_val @ V[:, :best_n]
                    
                    corrs = []
                    for k in range(best_n):
                        rk = np.corrcoef(X_scores[:, k], Z_scores[:, k])[0, 1]
                        if not np.isnan(rk):
                            corrs.append(rk)
                    
                    if len(corrs) > 0:
                        cv_scores.append(np.mean(corrs))
                except:
                    continue
            
            if len(cv_scores) > 0:
                mean_score = np.mean(cv_scores)
                results_grid.append((penaltyu, penaltyv, mean_score))
                
                if mean_score > best_score:
                    best_score = mean_score
                    best_penaltyu = penaltyu
                    best_penaltyv = penaltyv
        
        if verbose:
            print(f"Best penaltyu: {best_penaltyu:.3f}")
            print(f"Best penaltyv: {best_penaltyv:.3f}")
            print(f"Best CV score: {best_score:.4f}\n")
    else:
        if penaltyu is None or penaltyv is None:
            raise ValueError("penaltyu and penaltyv must be specified when find_best_penalty=False")
        best_penaltyu = penaltyu
        best_penaltyv = penaltyv
    
    # 3. 전체 데이터에 Sparse CCA 적용
    if verbose:
        print("=" * 50)
        print("Applying Sparse CCA with best parameters...")
        print(f"penaltyu = {best_penaltyu:.3f}, penaltyv = {best_penaltyv:.3f}, n_components = {best_n}")
        print("=" * 50)
    
    U, V, D = pmd(X.T @ Z, K=best_n,
                 penaltyu=best_penaltyu,
                 penaltyv=best_penaltyv,
                 standardize=False)
    
    X_scores = X @ U[:, :best_n]
    Z_scores = Z @ V[:, :best_n]
    
    # 각 컴포넌트의 canonical correlation 계산
    sparse_cors = []
    for k in range(best_n):
        rk = np.corrcoef(X_scores[:, k], Z_scores[:, k])[0, 1]
        sparse_cors.append(rk)
        if verbose:
            print(f"Sparse CCA Component {k+1}: correlation = {rk:.4f}")
    
    if verbose:
        print(f"\nAverage correlation: {np.mean(sparse_cors):.4f}\n")
    
    # 4. Weights 확인
    if X_cols is None:
        X_cols = [f'X_{i}' for i in range(X.shape[1])]
    if Z_cols is None:
        Z_cols = [f'Z_{i}' for i in range(Z.shape[1])]
    
    X_weights_dict = {}
    Z_weights_dict = {}
    
    for k in range(best_n):
        x_weights = pd.Series(U[:, k], index=X_cols)
        z_weights = pd.Series(V[:, k], index=Z_cols)
        
        if verbose:
            print(f"=== Component {k+1} ===")
            print("\nX Weights:")
            print(x_weights.sort_values(key=abs, ascending=False))
            print(f"Non-zero weights: {(x_weights != 0).sum()} / {len(x_weights)}")
            print("\nZ Weights:")
            print(z_weights.sort_values(key=abs, ascending=False))
            print(f"Non-zero weights: {(z_weights != 0).sum()} / {len(z_weights)}\n")
        
        X_weights_dict[f'Component_{k+1}'] = x_weights
        Z_weights_dict[f'Component_{k+1}'] = z_weights
    
    # 5. 시각화
    if visualize:
        fig, axes = plt.subplots(1, best_n, figsize=(6*best_n, 5))
        if best_n == 1:
            axes = [axes]
        
        for k in range(best_n):
            axes[k].scatter(X_scores[:, k], Z_scores[:, k], alpha=0.6)
            axes[k].set_xlabel(f'X Canonical Score (Component {k+1})')
            axes[k].set_ylabel(f'Z Canonical Score (Component {k+1})')
            axes[k].set_title(f'Component {k+1}: r = {sparse_cors[k]:.4f}')
            axes[k].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    # 6. Permutation Test
    permutation_pvalue = None
    if do_permutation_test:
        if verbose:
            print("\n" + "=" * 50)
            print("Permutation Test for Statistical Significance")
            print("=" * 50)
        
        def sparse_cca_statistic(X, Z, penaltyu, penaltyv, n_components):
            """Sparse CCA를 수행하고 canonical correlation의 합을 반환"""
            try:
                U, V, D = pmd(X.T @ Z, K=n_components,
                             penaltyu=penaltyu, penaltyv=penaltyv, standardize=False)
                
                X_scores = X @ U[:, :n_components]
                Z_scores = Z @ V[:, :n_components]
                
                corrs = []
                for k in range(n_components):
                    rk = np.corrcoef(X_scores[:, k], Z_scores[:, k])[0, 1]
                    if not np.isnan(rk):
                        corrs.append(rk)
                
                stat = np.sum(np.array(corrs)**2) if len(corrs) > 0 else 0.0
                return stat
            except:
                return 0.0
        
        r_obs = sparse_cca_statistic(X, Z, best_penaltyu, best_penaltyv, best_n)
        if verbose:
            print(f"Observed statistic (sum of squared correlations): {r_obs:.6f}")
            print(f"Running {n_permutations} permutations...")
        
        rng = np.random.default_rng(random_state)
        r_perm = []
        
        for i in range(n_permutations):
            idx = rng.permutation(X.shape[0])
            X_perm = X[idx, :]
            r_perm.append(sparse_cca_statistic(X_perm, Z, best_penaltyu, best_penaltyv, best_n))
            
            if verbose and (i + 1) % 100 == 0:
                print(f"  Completed {i+1}/{n_permutations} permutations...")
        
        r_perm = np.array(r_perm)
        # permutation_pvalue = np.mean(r_perm >= r_obs)
        permutation_pvalue = (np.sum(r_perm >= r_obs) + 1) / (n_permutations + 1)
        
        if verbose:
            print(f"\nPermutation p-value: {permutation_pvalue:.4f}")
            print(f"95% percentile of permuted statistics: {np.percentile(r_perm, 95):.6f}")
            print(f"99% percentile of permuted statistics: {np.percentile(r_perm, 99):.6f}")
        
        if visualize:
            plt.figure(figsize=(10, 6))
            plt.hist(r_perm, bins=50, alpha=0.7, edgecolor='black', label='Permuted statistics')
            plt.axvline(r_obs, color='red', linestyle='--', linewidth=2, label=f'Observed ({r_obs:.4f})')
            plt.axvline(np.percentile(r_perm, 95), color='orange', linestyle=':', linewidth=2, label='95th percentile')
            plt.xlabel('Test Statistic (Sum of Squared Correlations)')
            plt.ylabel('Frequency')
            plt.title(f'Permutation Test Distribution\np-value = {permutation_pvalue:.4f}')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.show()
    
    # 결과 반환
    results = {
        'best_n': best_n,
        'best_penaltyu': best_penaltyu,
        'best_penaltyv': best_penaltyv,
        'correlations': sparse_cors,
        'X_weights': X_weights_dict,
        'Z_weights': Z_weights_dict,
        'X_scores': X_scores,
        'Z_scores': Z_scores,
        'permutation_pvalue': permutation_pvalue
    }
    
    return results

In [5]:
def run_plsr(
    X,
    y,
    n_components=1,
    x_cols=None,
    scale=True,
    verbose=True,
    do_permutation=True,
    n_permutations=1000,
    cv_splits=5,
    random_state=42,
):

    if hasattr(X, "values"):
        if x_cols is None:
            x_cols = list(X.columns)
        X_mat = X.values
    else:
        X_mat = np.asarray(X)
        if x_cols is None:
            x_cols = [f"X{i+1}" for i in range(X_mat.shape[1])]

    # y를 2D로 맞추기
    if hasattr(y, "values"):
        y_mat = np.asarray(y).reshape(-1, 1) if y.ndim == 1 else y.values
    else:
        y_arr = np.asarray(y)
        y_mat = y_arr.reshape(-1, 1) if y_arr.ndim == 1 else y_arr

    # 스케일링
    if scale:
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        X_scaled = scaler_X.fit_transform(X_mat)
        y_scaled = scaler_y.fit_transform(y_mat)
    else:
        X_scaled = X_mat
        y_scaled = y_mat

    # PLS 적합
    pls = PLSRegression(n_components=n_components)
    pls.fit(X_scaled, y_scaled)

    # 예측 및 상관계수
    y_pred = pls.predict(X_scaled)
    r, p = pearsonr(y_scaled[:, 0], y_pred[:, 0])

    # Weights만 확인
    weights = pd.Series(pls.x_weights_[:, 0], index=x_cols).sort_values(key=abs, ascending=False)

    if verbose:
        print(f"PLSR Component 1과 y의 correlation: {r:.4f} (p={p:.4f})")
        print("\n=== PLSR Weights ===")
        print(weights)

    # Permutation test
    perm_p = None
    cv_score = None
    if do_permutation:
        try:
            pls_perm = PLSRegression(n_components=n_components)
            cv_score, perm_scores, perm_p = permutation_test_score(
                pls_perm,
                X_scaled,
                y_scaled.ravel(),
                scoring="r2",
                n_permutations=n_permutations,
                cv=cv_splits,
                random_state=random_state,
            )
            if verbose:
                print(f"\nPermutation test (R2) -> CV R2: {cv_score:.4f}, p_perm: {perm_p:.4f}")
        except Exception as e:
            if verbose:
                print(f"Permutation test failed: {e}")

    results = {
        "pls": pls,
        "r": r,
        "p": p,
        "weights": weights,
        "y_true_scaled": y_scaled[:, 0],
        "y_pred_scaled": y_pred[:, 0],
        "perm_p": perm_p,
        "cv_r2": cv_score,
    }

    return results

## 수정사항

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import svd
from statsmodels.multivariate.cancorr import CanCorr
from sparsecca import pmd
import seaborn as sns
import pingouin as pg
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import permutation_test_score
from scipy.stats import pearsonr
from impyute.imputation.cs import mice
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import missingno as msno
import miceforest as mf
from sklearn.cross_decomposition import CCA
from sklearn.model_selection import KFold
import itertools

path = 'C:\\Users\\nva_kist\\Desktop\\minsun\\KIST\\BioMarker\\Merge_v2.xlsx'
df = pd.read_excel(path)

cols = ['B',	'CD11b','HLADR','NK','NKT1','T',
        'NKT2',	'MAIT'	,'gd1T','gd2T',	'ConvT%',
'CD8+T','CD4+T','CD8+Treg','CD4+Treg','CD20T','CD4trm','CD8trm','MAITtrm','GD1Ttrm',	
'GD2Ttrm','CD4grzk', 'CD8grzk','MAITgrzK','GD1TgrzK','GD2TgrzK','Transitional B',
'Memory B','Early Plasma B ','Plasma B','ABC'
]

# % 제거하기
df[cols] = (df[cols]
            .replace(r'^\s*$', pd.NA, regex=True) 
            .replace({',': ''}, regex=True)
            .replace({'%': ''}, regex=True)   
            .apply(pd.to_numeric, errors='coerce')
            *100)      

# unnamed 인 column 제거 
df = df.loc[:, ~df.columns.str.contains(r'^Unnamed')]

In [7]:
#Sex column 0,1로 바꾸기
df['Sex_01'] = df['Sex'].map({'F': 0, 'M': 1})
print(df['Sex_01'].head())

0    0
1    1
2    1
3    0
4    1
Name: Sex_01, dtype: int64


### PCA로 column 추가하기

In [8]:
# PCA로 column 추가
from sklearn.decomposition import PCA

pca_groups = {
    "CD8":  ['CD8+T', 'CD8trm', 'CD8grzk'],
    "CD8_1":['CD8trm', 'CD8grzk'],
    "CD8_2":['CD8+T', 'CD8trm'],
    "CD8_3":['CD8+T', 'CD8grzk'],

    "CD4":   ['CD4+T', 'CD4trm', 'CD4grzk'],
    "CD4_1": ['CD4trm', 'CD4grzk'],
    "CD4_2": ['CD4+T', 'CD4trm'],
    "CD4_3": ['CD4+T', 'CD4grzk'],
    
    "MAIT":   ['MAIT', 'MAITtrm', 'MAITgrzK'],
    "MAIT_1": ['MAITtrm', 'MAITgrzK'],
    "MAIT_2": ['MAIT', 'MAITtrm'],
    "MAIT_3": ['MAIT', 'MAITgrzK'],

    "GD1T":   ['gd1T', 'GD1Ttrm', 'GD1TgrzK'],
    "GD1T_1": ['GD1Ttrm', 'GD1TgrzK'],
    "GD1T_2": ['gd1T', 'GD1Ttrm'],
    "GD1T_3": ['gd1T', 'GD1TgrzK'],

    "GD2T":   ['gd2T', 'GD2Ttrm', 'GD2TgrzK'],
    "GD2T_1": ['GD2Ttrm', 'GD2TgrzK'],
    "GD2T_2": ['gd2T', 'GD2Ttrm'],
    "GD2T_3": ['gd2T', 'GD2TgrzK'],

}

pca_info = {}

df = df.copy()

def add_pc1_score(df, cols, out_col):
    # 컬럼 존재 체크
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"[{out_col}] df2에 없는 컬럼이 있어요: {missing}")

    # 결측 없는 행만 PCA 계산
    tmp = df[cols].dropna()
    if tmp.shape[0] < 3:
        # 표본이 너무 적으면 그냥 NaN 채우고 메타만 남김
        df[out_col] = np.nan
        return {"explained_var_pc1": np.nan, "loadings_pc1": None, "n_used": int(tmp.shape[0])}

    # 표준화 -> PCA(PC1)
    Z = StandardScaler().fit_transform(tmp.values)
    pca = PCA(n_components=1)
    pc1 = pca.fit_transform(Z).ravel()

    # PC1 부호는 임의라서, 해석 편하게 "첫 변수"와 양의 상관이 되도록 정렬(원하면 바꾸셔도 됨)
    if np.corrcoef(pc1, Z[:, 0])[0, 1] < 0:
        pc1 = -pc1
        loadings = -pca.components_[0]
    else:
        loadings = pca.components_[0]

    # df에 넣기: 계산된 행만 채우고 나머지는 NaN 유지
    s = pd.Series(np.nan, index=df.index)
    s.loc[tmp.index] = pc1
    df[out_col] = s

    info = {
        "explained_var_pc1": float(pca.explained_variance_ratio_[0]),
        "loadings_pc1": dict(zip(cols, loadings)),
        "n_used": int(tmp.shape[0])
    }
    return info

for gname, cols in pca_groups.items():
    out_col = f"{gname}_PC1"
    info = add_pc1_score(df, cols, out_col)

# 확인
print(df[[f"{k}_PC1" for k in pca_groups.keys()]].head())

    CD8_PC1  CD8_1_PC1  CD8_2_PC1  CD8_3_PC1   CD4_PC1  CD4_1_PC1  CD4_2_PC1  \
0       NaN        NaN        NaN        NaN       NaN        NaN        NaN   
1  1.270096   0.795941   1.864011   1.916927 -0.890699   0.267301  -0.968928   
2       NaN        NaN        NaN        NaN       NaN        NaN        NaN   
3       NaN        NaN        NaN        NaN       NaN        NaN        NaN   
4 -2.447762  -2.941932  -0.139552  -0.317396 -3.121833  -2.890633  -2.414740   

   CD4_3_PC1  MAIT_PC1  MAIT_1_PC1  MAIT_2_PC1  MAIT_3_PC1  GD1T_PC1  \
0        NaN       NaN         NaN         NaN         NaN       NaN   
1  -1.359545 -0.493880   -0.309122   -1.019099    0.590695 -0.979231   
2        NaN       NaN         NaN         NaN         NaN       NaN   
3        NaN       NaN         NaN         NaN         NaN       NaN   
4  -2.417959  0.496095   -0.233762    0.920678   -1.052133 -1.116262   

   GD1T_1_PC1  GD1T_2_PC1  GD1T_3_PC1  GD2T_PC1  GD2T_1_PC1  GD2T_2_PC1  \
0         N

In [9]:
df1 = df.iloc[:22]
df2 = df.iloc[22:]
print(df1.shape, df2.shape)

(22, 86) (22, 86)


In [10]:
df.columns

Index(['ID', 'Sex', 'Age', 'BMI', 'AST', 'ALT', 'ALP', 'Bilirubin', 'Protein',
       'Albumin', 'PT', 'APTT', 'Cholesterol', 'TG', 'LDL', 'HDL', 'R_Glucose',
       'F_Glucose', 'HbA1c', 'BUN', 'Cr', 'GFR(CKD_EPI)', 'CrCl', 'Amylase',
       'CRP', 'B', 'CD11b', 'HLADR', 'NK', 'NKT1', 'T', 'NKT2', 'MAIT', 'gd1T',
       'gd2T', 'ConvT%', 'CD8+T', 'CD4+T', 'CD8+Treg', 'CD4+Treg', 'CD20T',
       'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm', 'CD4grzk',
       'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK', 'Transitional B',
       'Memory B', 'Early Plasma B ', 'Plasma B', 'ABC', 'CD11c+CD206-',
       'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-', 'Dectin-1+1',
       'Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4', 'Gulucose', 'Sex_01',
       'CD8_PC1', 'CD8_1_PC1', 'CD8_2_PC1', 'CD8_3_PC1', 'CD4_PC1',
       'CD4_1_PC1', 'CD4_2_PC1', 'CD4_3_PC1', 'MAIT_PC1', 'MAIT_1_PC1',
       'MAIT_2_PC1', 'MAIT_3_PC1', 'GD1T_PC1', 'GD1T_1_PC1', 'GD1T_2_PC1',
       'GD1T_3_PC1', 'GD2T_PC1', 'GD2

### Task 1 . 통제변수 추가


통제변수 기존에는 BMI 성별만

추가: 콜레스테롤, gulucose + KUMC AMC동시에 가지고 있는 column

In [53]:
all_cols = [ 'B', 'CD11b', 'HLADR', 'NK', 'NKT1', 'T', 'NKT2', 'MAIT', 'gd1T',
       'gd2T', 'ConvT%', 'CD8+T', 'CD4+T', 'CD8+Treg', 'CD4+Treg', 'CD20T',
       'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm', 'CD4grzk',
       'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK', 'Transitional B',
       'Memory B', 'Early Plasma B ', 'Plasma B', 'ABC', 'CD11c+CD206-',
       'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-', 'Dectin-1+1',
       'Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4']

rows = []
for c in all_cols:
    # 컬럼별 필요한 값만 결측치 제거
    df_c = df.dropna(subset=['Age', 'BMI', c]).copy()

    if df_c.shape[0] < 6:
        rows.append({"var": c, "r": None, "p": None, "n": df_c.shape[0], "note": "SKIP (n too small)"})
        continue

    res = pg.partial_corr(
        data=df_c, x='Age', y=c, covar=['BMI', 'Sex_01'], method='pearson'
    )

    rows.append({
        "var": c,
        "r": float(res['r'].iloc[0]),
        "p": float(res['p-val'].iloc[0]),
        "n": int(res['n'].iloc[0]),
        "note": ""
    })

out = pd.DataFrame(rows)

out_sorted = out.sort_values(by="p", ascending=True, na_position="last").reset_index(drop=True)

for _, row in out_sorted.iterrows():
    if pd.isna(row["p"]):
        print(f"[SKIP] Age vs {row['var']}: n={row['n']} {row['note']}".strip())
    else:
        print(f"Age vs {row['var']} | r={row['r']:.3f}, p={row['p']:.4g}, n={row['n']}")


Age vs ConvT% | r=0.496, p=0.003334, n=35
Age vs NKT2 | r=-0.422, p=0.01442, n=35
Age vs T | r=0.348, p=0.02391, n=44
Age vs HLADR | r=-0.324, p=0.03647, n=44
Age vs CD11c-CD206+ | r=0.298, p=0.05519, n=44
Age vs CD11b | r=-0.262, p=0.09348, n=44
Age vs gd2T | r=-0.286, p=0.1061, n=35
Age vs gd1T | r=-0.277, p=0.1191, n=35
Age vs Plasma B | r=0.243, p=0.1213, n=44
Age vs CD20T | r=0.233, p=0.1372, n=44
Age vs NK | r=-0.203, p=0.1974, n=44
Age vs CD11c-CD206- | r=0.201, p=0.2018, n=44
Age vs Transitional B | r=-0.180, p=0.2542, n=44
Age vs MAIT | r=-0.199, p=0.2671, n=35
Age vs CD8+Treg | r=0.194, p=0.2795, n=35
Age vs MAITtrm | r=-0.181, p=0.3146, n=35
Age vs GD1TgrzK | r=0.180, p=0.3688, n=29
Age vs MAITgrzK | r=0.149, p=0.4067, n=35
Age vs CD8+T | r=0.149, p=0.4078, n=35
Age vs CD4+Treg | r=0.148, p=0.4117, n=35
Age vs CD11c+CD206- | r=-0.127, p=0.4246, n=44
Age vs NKT1 | r=-0.120, p=0.4472, n=44
Age vs GD2TgrzK | r=0.137, p=0.4482, n=35
Age vs Early Plasma B  | r=0.120, p=0.4497, n=

In [54]:
import numpy as np
import pandas as pd

from plotnine import (
    ggplot, aes, geom_point, geom_hline, geom_vline, geom_text,
    labs, theme_bw, theme, element_text,
    scale_color_manual, scale_size_area, scale_x_log10,
    guides
)

# =========================
# 0) 데이터 준비 (안전장치 포함)
# =========================
plot_df = out_sorted.copy()

# p, r 숫자형 강제 (문자/이상치 방지)
plot_df["p"] = pd.to_numeric(plot_df["p"], errors="coerce")
plot_df["r"] = pd.to_numeric(plot_df["r"], errors="coerce")

# 필요한 값 결측 제거 + log 축 대비(p>0)
plot_df = plot_df.dropna(subset=["p", "r"]).copy()
plot_df = plot_df[plot_df["p"] > 0].copy()

# 유의성(핑크/하늘)
plot_df["sig"] = np.where(plot_df["p"] < 0.05, "p<0.05", "ns")

# 부호(테두리)
plot_df["corr_dir"] = np.where(plot_df["r"] < 0, "negative", "positive")

# |r| 클수록 점 크게
plot_df["abs_r"] = plot_df["r"].abs()

# 라벨: p<0.05만
label_df = plot_df[plot_df["p"] < 0.05].copy()

# =========================
# 1) Plot
# =========================
p = (
    ggplot(plot_df, aes(x="p", y="r"))
    + geom_hline(yintercept=0, linetype="dashed", alpha=0.6)
    + geom_vline(xintercept=0.05, linetype="dashed", alpha=0.6)
    + scale_x_log10()  # p-value를 log 스케일로 펼쳐서 보기 좋게

    # 1) 채움(유의성): 꽉 찬 원 (stroke=0으로 테두리 제거)
    + geom_point(aes(color="sig", size="abs_r"),
                 alpha=0.85, shape="o", stroke=0)

    # 2) 테두리(부호): 빈 원(테두리만) 덮기
    + geom_point(aes(color="corr_dir", size="abs_r"),
                 alpha=0.95, shape="o", fill=None, stroke=1.8)

    # 색 매핑 (요청대로 유지)
    + scale_color_manual(values={
        "ns": "#F8766D",
        "p<0.05": "#00BFC4",
        "negative": "#00BFC4",
        "positive": "#F8766D"
    })

    # 점 크기 차이
    + scale_size_area(max_size=25)

    # p<0.05 라벨
    + geom_text(
        label_df, aes(label="var"),
        size=8, va="bottom", ha="left",
        nudge_y=0.01, nudge_x=0.02,
        show_legend=False
    )

    + labs(
        title="Partial correlation: Age vs immune features (covar: BMI)",
        x="p-value",
        y="partial r"
    )

    # ✅ 범례 완전 삭제 (color, size 전부)
    + guides(color="none", size="none")

    + theme_bw()
    + theme(
        figure_size=(10, 6),
        axis_text=element_text(size=10),
        axis_title=element_text(size=12),
    )
)

# =========================
# 2) 저장
# =========================
p.save(r"C:/Users/nva_kist/Desktop/minsun/KIST/BioMarker/1.png", dpi=300, verbose=False)
print("Saved:", r"C:/Users/nva_kist/Desktop/minsun/KIST/BioMarker/1.png")


Saved: C:/Users/nva_kist/Desktop/minsun/KIST/BioMarker/1.png


##### Age와 관련있는 면역세포 : HLADR, T, NKT2, ConvT, (CD11c-CD206+)

#####  통제변수 다르게 하며 결과 분석하기 

In [57]:
import statsmodels.api as sm

def partial_corr_df(df, x, y, covars):
    cols = [x, y] + list(covars)
    d = df[cols].dropna().copy()
    n = len(d)
    if n < 6:
        return {"n": n, "r": np.nan, "p": np.nan}

    Xc = sm.add_constant(d[covars]) if len(covars) > 0 else None

    if len(covars) == 0:
        r, p = pearsonr(d[x], d[y])
        return {"n": n, "r": r, "p": p}

    rx = sm.OLS(d[x], Xc).fit().resid
    ry = sm.OLS(d[y], Xc).fit().resid
    r, p = pearsonr(rx, ry)
    return {"n": n, "r": r, "p": p}

def run_models(df, x, y_list, cov_sets, title=None, fix_n=False):
    # fix_n=True면 모든 모델에서 동일한 complete-case로 고정(비교 공정)
    if fix_n:
        all_covs = sorted({c for cov in cov_sets.values() for c in cov})
        base_cols = [x] + y_list + all_covs
        df_use = df[base_cols].dropna().copy()
    else:
        df_use = df

    rows = []
    if title:
        print(f"\n========== {title} ==========")

    for y in y_list:
        line = []
        for mname, covars in cov_sets.items():
            out = partial_corr_df(df_use, x, y, covars)
            rows.append({"y": y, "model": mname, **out})
            # 스크린샷 스타일 출력
            line.append(f"{mname}: r={out['r']:.3f}, p={out['p']:.4f}, n={out['n']}")
        print(f"{x} vs {y} | " + " | ".join(line))

    res = pd.DataFrame(rows)

    # 변화량 계산: 기준 모델 하나 잡기
    pivot_r = res.pivot(index="y", columns="model", values="r")
    base = list(cov_sets.keys())[0]  # 첫 모델을 기준으로(원하면 바꾸세요)
    for m in cov_sets.keys():
        pivot_r[f"delta_r({m}-{base})"] = pivot_r[m] - pivot_r[base]

    return res, pivot_r

# ===== 사용 예시 =====
# 1) 모델 정의
cov_sets = {
    "M0_sex": ["Sex_01"],
    "M1_BMI": ["BMI"],
    "M2_sex_BMI": ["Sex_01", "BMI"],
    "M3_sex_BMI_c": ["Sex_01", "BMI", "Cholesterol"],
    "M4_sex_BMI_g": ["Sex_01", "BMI", "Gulucose"],
}

# 2) 분석할 면역 변수 리스트
tsubset_vars = ['B', 'CD11b', 'HLADR', 'NK', 'NKT1', 'T', 'NKT2', 'MAIT', 'gd1T',
       'gd2T', 'ConvT%', 'CD8+T', 'CD4+T', 'CD8+Treg', 'CD4+Treg', 'CD20T',
       'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm', 'CD4grzk',
       'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK', 'Transitional B',
       'Memory B', 'Early Plasma B ', 'Plasma B', 'ABC', 'CD11c+CD206-',
       'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-', 'Dectin-1+1',
       'Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4', 'CD8_PC1', 'CD4_PC1', 'MAIT_PC1', 'GD1T_PC1', 'GD2T_PC1']

# 3) 실행
res_long, res_delta = run_models(df, x="Age", y_list=tsubset_vars, cov_sets=cov_sets,
                                 title="면역세포", fix_n=True)

# res_long: (y, model, r, p, n) long-format
# res_delta: 모델별 r + delta_r 테이블
print("\n--- delta table (r 변화) ---")
print(res_delta)

# p<=0.1인 결과만 뽑아서 보기
sig = (
    res_long
    .dropna(subset=["p"])
    .query("p <= 0.1")
    .sort_values(["p", "model", "y"])
)

print(f"Significant (p<=0.1) rows: {len(sig)}")
print(sig[["y", "model", "n", "r", "p"]].to_string(index=False))



========== 면역세포 ==========
Age vs B | M0_sex: r=0.010, p=0.9595, n=27 | M1_BMI: r=0.031, p=0.8790, n=27 | M2_sex_BMI: r=0.049, p=0.8096, n=27 | M3_sex_BMI_c: r=-0.054, p=0.7888, n=27 | M4_sex_BMI_g: r=0.023, p=0.9089, n=27
Age vs CD11b | M0_sex: r=-0.120, p=0.5517, n=27 | M1_BMI: r=-0.089, p=0.6582, n=27 | M2_sex_BMI: r=-0.083, p=0.6803, n=27 | M3_sex_BMI_c: r=-0.041, p=0.8379, n=27 | M4_sex_BMI_g: r=-0.064, p=0.7505, n=27
Age vs HLADR | M0_sex: r=-0.324, p=0.0993, n=27 | M1_BMI: r=-0.319, p=0.1054, n=27 | M2_sex_BMI: r=-0.304, p=0.1234, n=27 | M3_sex_BMI_c: r=-0.229, p=0.2496, n=27 | M4_sex_BMI_g: r=-0.278, p=0.1601, n=27
Age vs NK | M0_sex: r=-0.390, p=0.0440, n=27 | M1_BMI: r=-0.348, p=0.0752, n=27 | M2_sex_BMI: r=-0.405, p=0.0363, n=27 | M3_sex_BMI_c: r=-0.357, p=0.0679, n=27 | M4_sex_BMI_g: r=-0.402, p=0.0375, n=27
Age vs NKT1 | M0_sex: r=-0.135, p=0.5034, n=27 | M1_BMI: r=-0.123, p=0.5405, n=27 | M2_sex_BMI: r=-0.124, p=0.5392, n=27 | M3_sex_BMI_c: r=-0.070, p=0.7293, n=27 | M4_

In [70]:
import numpy as np
import pandas as pd
from plotnine import *

# =========================
# 0) 데이터 준비
# =========================
plot_df = res_long.dropna(subset=["r", "p"]).copy()

# (선택) 숫자형 강제 변환
plot_df["r"] = pd.to_numeric(plot_df["r"], errors="coerce")
plot_df["p"] = pd.to_numeric(plot_df["p"], errors="coerce")
plot_df = plot_df.dropna(subset=["r", "p"]).copy()

# ✅ x축 라벨 매핑
label_map = {
    "M0_sex": "sex",
    "M1_BMI": "BMI",
    "M2_sex_BMI": "sex + BMI",
    "M3_sex_BMI_c": "sex + BMI + Cholesterol",
    "M4_sex_BMI_g": "sex + BMI + Glucose",
}
plot_df["model"] = plot_df["model"].map(label_map)

# ✅ 원하는 순서 고정
new_order = ["sex", "BMI", "sex + BMI", "sex + BMI + Cholesterol", "sex + BMI + Glucose"]
plot_df["model"] = pd.Categorical(plot_df["model"], categories=new_order, ordered=True)

# =========================
# 1) 유의성 구간 나누기
# =========================
sig_df = plot_df[plot_df["p"] < 0.05].copy()
mid_df = plot_df[(plot_df["p"] >= 0.05) & (plot_df["p"] < 0.1)].copy()
# p>=0.1 은 표시 안 함

# =========================
# 2) Heatmap + 표시 오버레이
# =========================
p_heat = (
    ggplot(plot_df, aes(x="model", y="y", fill="r"))
    + geom_tile()

    # ✅ 0.05 <= p < 0.1 : 작은 점(·)
    + geom_text(
        mid_df,
        aes(x="model", y="y", label="'·'"),
        inherit_aes=False,
        size=15,
        alpha=0.7
    )

    # ✅ p < 0.05 : 별(*)
    + geom_text(
        sig_df,
        aes(x="model", y="y", label="'*'"),
        inherit_aes=False,
        size=15,
        alpha=0.95
    )

    + scale_fill_gradient2(low="#00BFC4", mid="white", high="#F8766D")
    + labs(
        title="Age & Immune cell by covariate (*: p<0.05, ·: 0.05≤p<0.1)",
        x="Covariates (model)",
        y="Immune feature",
        fill="r"
    )
    + theme_bw()
    + theme(
        figure_size=(12, 12),
        axis_text_y=element_text(size=7),
        axis_text_x=element_text(size=9)
    )
)

print(p_heat)

# =========================
# 3) 저장
# =========================
p_heat.save("heatmap_r_by_model_pgroup.png", dpi=300, verbose=False)
print("Saved: heatmap_r_by_model_pgroup.png")


<ggplot: (1200 x 1200)>
Saved: heatmap_r_by_model_pgroup.png


In [71]:
from plotnine import *

rank_df = (
    res_long.dropna(subset=["p","r"])
    .assign(p=lambda d: pd.to_numeric(d["p"], errors="coerce"))
    .dropna(subset=["p"])
    .query("p > 0")
    .copy()
)

# ✅ 모델 라벨 매핑(이전에 쓰던거)
label_map = {
    "M0_sex": "sex",
    "M1_BMI": "BMI",
    "M2_sex_BMI": "sex + BMI",
    "M3_sex_BMI_c": "sex + BMI + Cholesterol",
    "M4_sex_BMI_g": "sex + BMI + Glucose",
}
rank_df["model"] = rank_df["model"].map(label_map)

# ✅ -log10(p) (클수록 더 유의)
rank_df["mlog10p"] = -np.log10(rank_df["p"])

# 예: 모델별 Top 10
topN = 10
top_df = (rank_df.sort_values(["model","p"])
                  .groupby("model")
                  .head(topN)
                  .copy())

p_rank = (
    ggplot(top_df, aes(x="reorder(y, mlog10p)", y="mlog10p", fill="r"))
    + geom_col()
    + coord_flip()
    + facet_wrap("~model", scales="free_y")
    + scale_fill_gradient2(low="#00BFC4", mid="white", high="#F8766D")
    + labs(title=f"Top {topN} smallest p-values per model", x="Immune feature", y="-log10(p)", fill="r")
    + theme_bw()
    + theme(figure_size=(12, 8), axis_text=element_text(size=8))
)

print(p_rank)
p_rank.save("p_rank_topN_by_model.png", dpi=300, verbose=False)


<ggplot: (1200 x 800)>


## Task2 - (1)KUMC 분석

In [ ]:
# 하나씩 test해보자

# df1 - KUMC, 결측치 제거 후 sCCA 진행 

# 1) 그룹 정의
X_groups = {
    "Metabolic": ['R_Glucose', 'Cholesterol'],
    "Liver": ['AST', 'ALT', 'ALP', 'Bilirubin', 'Protein', 'Albumin'],
    "Liver1" : ['AST', 'ALT', 'ALP'],
    "Liver2" : ['Bilirubin', 'Protein', 'Albumin'],
    "Kidney": ['BUN', 'Cr'],
    "ETC": ['Amylase', 'CRP'],
    "All": ['AST', 'ALT', 'ALP', 'Bilirubin', 'Protein', 'Albumin', 
               'Cholesterol', 'R_Glucose', 'BUN', 'Cr', 'Amylase', 'CRP']
               }

# 박사님 그룹
Z_groups = {"GD2T": ['gd2T','GD2Ttrm', 'GD2TgrzK'],
            "GD2T_1" : ['GD2T_1_PC1', 'gd2T'],
            "GD2T_2": ['GD2Ttrm', 'GD2TgrzK'],
            "GD2T_3": ['gd2T']}

rows = []
for x_name, X_cols in X_groups.items():
    for z_name, Z_cols in Z_groups.items():
        use_cols = X_cols + Z_cols

        # 결측치 있는 행 제거 
        df_clean = df1.dropna(subset=use_cols).copy()
        n = df_clean.shape[0]

        if n < 6:
            rows.append({
                "X_group": x_name, "Z_group": z_name,
                "n": n, "r1": np.nan, "p_perm": np.nan,
                "status": "SKIP(n too small)"
            })
            continue

        X = df_clean[X_cols]
        Z = df_clean[Z_cols]


        try:
            results = apply_sparse_cca(
                X, Z,
                X_cols=X_cols, Z_cols=Z_cols,
                verbose=False,
                visualize=False
            )

            r = results.get("correlations", None)
            p = results.get("permutation_pvalue", None)

            r1 = float(r[0]) if isinstance(r, (list, tuple, np.ndarray)) else float(r)

            rows.append({
                "X_group": x_name,
                "Z_group": z_name,
                "n": n,
                "r1": r1,
                "p_perm": p,
                "status": "OK"
            })

        except Exception as e:
            rows.append({
                "X_group": x_name, "Z_group": z_name,
                "n": n, "r1": np.nan, "p_perm": np.nan,
                "status": f"FAIL: {type(e).__name__}"
            })

res_df = pd.DataFrame(rows)
res_df_sorted = res_df.sort_values(["status", "p_perm"], ascending=[True, True])
print(res_df_sorted)

      X_group Z_group   n        r1    p_perm status
3       Liver    GD2T  12  0.838574  0.074925     OK
4       Liver  GD2T_1  12  0.826399  0.085914     OK
7      Liver1  GD2T_1  12  0.650609  0.242757     OK
6      Liver1    GD2T  12  0.652845  0.250749     OK
0   Metabolic    GD2T  11  0.604961  0.260739     OK
10     Liver2  GD2T_1  13  0.557880  0.311688     OK
17        ETC  GD2T_2  11  0.429867  0.388611     OK
18        All    GD2T  10  0.794259  0.407592     OK
11     Liver2  GD2T_2  13  0.432814  0.450549     OK
19        All  GD2T_1  10  0.667012  0.471528     OK
16        ETC  GD2T_1  11  0.426885  0.488511     OK
13     Kidney  GD2T_1  13  0.373213  0.590410     OK
9      Liver2    GD2T  13  0.443645  0.614386     OK
15        ETC    GD2T  11  0.417031  0.631369     OK
20        All  GD2T_2  10  0.716708  0.650350     OK
12     Kidney    GD2T  13  0.373213  0.672328     OK
1   Metabolic  GD2T_1  11  0.404530  0.692308     OK
8      Liver1  GD2T_2  12  0.370781  0.706294 

In [18]:
# df1 - KUMC, 결측치 제거 후 sCCA 진행 
# 전체 df1 & 면역세포

# 1) 그룹 정의
X_groups = {
    "Metabolic": ['R_Glucose', 'Cholesterol'],
    "Liver": ['AST', 'ALT', 'ALP', 'Bilirubin', 'Protein', 'Albumin'],
    "Liver1" : ['AST', 'ALT', 'ALP','Bilirubin'],
    "Liver2" : [ 'Protein', 'Albumin'],
    "Kidney": ['BUN', 'Cr'],
    "ETC": ['Amylase', 'CRP'],
    # "All": ['AST', 'ALT', 'ALP', 'Bilirubin', 'Protein', 'Albumin', 
    #            'Cholesterol', 'R_Glucose', 'BUN', 'Cr', 'Amylase', 'CRP']
               }

# 박사님 그룹
Z_groups = {
    "B" : ["Transitional B", "Memory B", "Early Plasma B ", "Plasma B", "ABC"],
    "T" : ['NKT2', 'MAIT', 'gd1T','gd2T', 'ConvT%'],
    "ConvT": ['CD8+T', 'CD4+T', 'CD8+Treg', 'CD4+Treg'],
    "PAN": ['B', 'CD11b', 'HLADR', 'NK', 'NKT1', 'T'], 
    "CD8":  ['CD8+T', 'CD8trm', 'CD8grzk'],
    "CD4":  ['CD4+T',  'CD4trm', 'CD4grzk'],
    "MAIT": ['MAIT', 'MAITtrm', 'MAITgrzK'],
    "CD20T":["CD20T"],
    "GD1T": ['gd1T','GD1Ttrm', 'GD1TgrzK'],
    "GD2T": ['gd2T','GD2Ttrm', 'GD2TgrzK'],
    "PCA" : ['CD8_PC1','CD4_PC1', 'MAIT_PC1', 'GD1T_PC1', 'GD2T_PC1'],
    "CD11c": ['CD11c+CD206-', 'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-', 'Dectin-1+1','Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4'],
    "Detectin":[ 'Dectin-1+1','Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4'],
    "CD11c_1" :['CD11c+CD206-', 'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-']
}


rows = []
for x_name, X_cols in X_groups.items():
    for z_name, Z_cols in Z_groups.items():
        use_cols = X_cols + Z_cols

        # 결측치 있는 행 제거 
        df_clean = df1.dropna(subset=use_cols).copy()
        n = df_clean.shape[0]

        if n < 6:
            rows.append({
                "X_group": x_name, "Z_group": z_name,
                "n": n, "r1": np.nan, "p_perm": np.nan,
                "status": "SKIP(n too small)"
            })
            continue

        X = df_clean[X_cols]
        Z = df_clean[Z_cols]


        try:
            results = apply_sparse_cca(
                X, Z,
                X_cols=X_cols, Z_cols=Z_cols,
                verbose=False,
                visualize=False
            )

            r = results.get("correlations", None)
            p = results.get("permutation_pvalue", None)

            r1 = float(r[0]) if isinstance(r, (list, tuple, np.ndarray)) else float(r)

            rows.append({
                "X_group": x_name,
                "Z_group": z_name,
                "n": n,
                "r1": r1,
                "p_perm": p,
                "status": "OK"
            })

        except Exception as e:
            rows.append({
                "X_group": x_name, "Z_group": z_name,
                "n": n, "r1": np.nan, "p_perm": np.nan,
                "status": f"FAIL: {type(e).__name__}"
            })

res_df = pd.DataFrame(rows)
res_df_sorted1 = res_df.sort_values(["status", "p_perm"], ascending=[True, True])
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
res_df_sorted1

,X_group,Z_group,n,r1,p_perm,status
16,Liver,ConvT,12,0.860993,0.006993,OK
56,Kidney,B,22,0.606120,0.020979,OK
82,ETC,Detectin,18,0.615581,0.022977,OK
42,Liver2,B,22,0.676248,0.024975,OK
29,Liver1,T,12,0.818518,0.032967,OK
74,ETC,CD8,11,0.867487,0.035964,OK
59,Kidney,PAN,22,0.687289,0.037962,OK
81,ETC,CD11c,18,0.615581,0.047952,OK
32,Liver1,CD8,12,0.837781,0.054945,OK
23,Liver,GD2T,12,0.838574,0.074925,OK


### Task2- ssca df2에 적용

In [19]:
#df2로 동일한 진행 - 결측치는 삭제하고 sparse cca, 간신장대사 나눠서 진행

# 1) df2용 그룹 정의 

X_groups = {
    "Liver":   ['AST', 'ALT', 'ALP', 'Bilirubin', 'Albumin', 'PT', 'APTT'],
    "Liver1" : ['AST', 'ALT', 'ALP', 'PT', 'APTT', 'Bilirubin'] , 
    "Liver2":["Albumin"],
    "Metabolic": ['TG', 'LDL',  'F_Glucose', 'HbA1c', 'Cholesterol','HDL'],
    "HDL": ['HDL'],
    "Metaboilic1":['TG', 'LDL',  'F_Glucose', 'HbA1c', 'Cholesterol'],
    "Kidney": ['BUN', 'Cr', 'GFR(CKD_EPI)', 'CrCl'],
    "Kidney1": ['BUN', 'Cr'],
    "Kidney2": ['GFR(CKD_EPI)', 'CrCl'],
}


# 박사님 그룹
Z_groups = {
    "B" : ["Transitional B", "Memory B", "Early Plasma B ", "Plasma B", "ABC"],
    "T" : ['NKT2', 'MAIT', 'gd1T','gd2T', 'ConvT%'],
    "ConvT": ['CD8+T', 'CD4+T', 'CD8+Treg', 'CD4+Treg'],
    "PAN": ['B', 'CD11b', 'HLADR', 'NK', 'NKT1', 'T'], 
    "CD8":  ['CD8+T', 'CD8trm', 'CD8grzk'],
    "CD4":  ['CD4+T',  'CD4trm', 'CD4grzk'],
    "MAIT": ['MAIT', 'MAITtrm', 'MAITgrzK'],
    "GD1T": ['gd1T','GD1Ttrm', 'GD1TgrzK'],
    "GD2T": ['gd2T','GD2Ttrm', 'GD2TgrzK'],
    "PCA" : ['CD8_PC1','CD4_PC1', 'MAIT_PC1', 'GD1T_PC1', 'GD2T_PC1'],
    "CD11c": ['CD11c+CD206-', 'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-', 'Dectin-1+1','Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4'],
    "Detectin":[ 'Dectin-1+1','Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4'],
    "CD11c_1" :['CD11c+CD206-', 'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-']
}


rows = []
for x_name, X_cols in X_groups.items():
    for z_name, Z_cols in Z_groups.items():
        use_cols = X_cols + Z_cols

        # 조합별 결측치 행 제거
        df_clean = df2.dropna(subset=use_cols).copy()
        n = df_clean.shape[0]

        if n < 6:
            rows.append({
                "X_group": x_name, "Z_group": z_name,
                "n": n, "r1": np.nan, "p_perm": np.nan,
                "status": "SKIP(n too small)"
            })
            continue

        X = df_clean[X_cols]
        Z = df_clean[Z_cols]

        try:
            results = apply_sparse_cca(
                X, Z,
                X_cols=X_cols, Z_cols=Z_cols,
                verbose=False,
                visualize=False
            )

            r = results.get("correlations", None)
            p = results.get("permutation_pvalue", None)

            r1 = float(r[0]) if isinstance(r, (list, tuple, np.ndarray)) else float(r)

            rows.append({
                "X_group": x_name,
                "Z_group": z_name,
                "n": n,
                "r1": r1,
                "p_perm": p,
                "status": "OK"
            })

        except Exception as e:
            rows.append({
                "X_group": x_name, "Z_group": z_name,
                "n": n, "r1": np.nan, "p_perm": np.nan,
                "status": f"FAIL: {type(e).__name__}"
            })

res_df = pd.DataFrame(rows)

res_df_sorted2 = res_df.sort_values(["status", "p_perm"], ascending=[True, True])
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
res_df_sorted2

C:\Users\nva_kist\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\nva_kist\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\nva_kist\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\nva_kist\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide

,X_group,Z_group,n,r1,p_perm,status
34,Liver2,GD2T,22,0.718038,0.000999,OK
8,Liver,GD2T,22,0.775333,0.003996,OK
15,Liver1,ConvT,22,0.755233,0.003996,OK
2,Liver,ConvT,22,0.758536,0.005994,OK
110,Kidney2,MAIT,22,0.544622,0.045954,OK
55,HDL,PAN,22,0.577672,0.046953,OK
77,Metaboilic1,CD11c_1,22,0.659663,0.067932,OK
71,Metaboilic1,MAIT,22,0.639751,0.080919,OK
84,Kidney,MAIT,22,0.544622,0.090909,OK
91,Kidney1,B,22,0.549385,0.092907,OK


## scca에서 통제한 후에 

In [16]:
import statsmodels.api as sm

# -----------------------------
# 0) residualize helper
# -----------------------------
def residualize_df(mat_df: pd.DataFrame, cov_df: pd.DataFrame) -> pd.DataFrame:
    """
    mat_df: (n x p) matrix dataframe (X or Z)
    cov_df: (n x k) covariates dataframe (BMI, Sex_01 etc.)
    return: residualized mat_df (same shape/cols)
    """
    Xc = sm.add_constant(cov_df.astype(float), has_constant="add")
    resid = {}
    for col in mat_df.columns:
        y = mat_df[col].astype(float).values
        fit = sm.OLS(y, Xc).fit()
        resid[col] = fit.resid
    return pd.DataFrame(resid, index=mat_df.index)

# -----------------------------
# 1) 그룹 정의 (원본 그대로)
# -----------------------------
X_groups = {
    "Metabolic": ['R_Glucose', 'Cholesterol'],
    "Liver": ['AST', 'ALT', 'ALP', 'Bilirubin', 'Protein', 'Albumin'],
    "Kidney": ['BUN', 'Cr'],
    "ETC": ['Amylase', 'CRP']}


Z_groups = {
    "B" : ["Transitional B", "Memory B", "Early Plasma B ", "Plasma B", "ABC"],
    "T" : ['NKT2', 'MAIT', 'gd1T','gd2T', 'ConvT%'],
    "ConvT": ['CD8+T', 'CD4+T', 'CD8+Treg', 'CD4+Treg'],
    "PAN": ['B', 'CD11b', 'HLADR', 'NK', 'NKT1', 'T'],
    "CD8":  ['CD8+T', 'CD8trm', 'CD8grzk'],
    "CD4":  ['CD4+T',  'CD4trm', 'CD4grzk'],
    "MAIT": ['MAIT', 'MAITtrm', 'MAITgrzK'],
    "CD20T":["CD20T"],
    "GD1T": ['gd1T','GD1Ttrm', 'GD1TgrzK'],
    "GD2T": ['gd2T','GD2Ttrm', 'GD2TgrzK'],
    "PCA" : ['CD8_PC1','CD4_PC1', 'MAIT_PC1', 'GD1T_PC1', 'GD2T_PC1'],
    "CD11c": ['CD11c+CD206-', 'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-',
              'Dectin-1+1','Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4'],
    "Detectin":[ 'Dectin-1+1','Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4'],
    "CD11c_1" :['CD11c+CD206-', 'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-']
}

# -----------------------------
# 2) covariates 지정
# -----------------------------
COVARS = ["BMI"]   # <- 요청한 컬럼명

# -----------------------------
# 3) 실행 (BMI, Sex_01 통제한 sCCA)
# -----------------------------
rows = []
for x_name, X_cols in X_groups.items():
    for z_name, Z_cols in Z_groups.items():

        use_cols = X_cols + Z_cols + COVARS

        # 결측치 있는 행 제거 (covariate까지 포함해서 complete-case)
        df_clean = df1.dropna(subset=use_cols).copy()
        n = df_clean.shape[0]

        if n < 6:
            rows.append({
                "X_group": x_name, "Z_group": z_name,
                "n": n, "r1": np.nan, "p_perm": np.nan,
                "status": "SKIP(n too small)"
            })
            continue

        # 원본 행렬
        X_raw = df_clean[X_cols]
        Z_raw = df_clean[Z_cols]
        cov = df_clean[COVARS].astype(float)

        # ✅ 통제: residualize
        X = residualize_df(X_raw, cov)
        Z = residualize_df(Z_raw, cov)

        try:
            results = apply_sparse_cca(
                X, Z,
                X_cols=X_cols, Z_cols=Z_cols,
                verbose=False,
                visualize=False
            )

            r = results.get("correlations", None)
            p = results.get("permutation_pvalue", None)
            r1 = float(r[0]) if isinstance(r, (list, tuple, np.ndarray)) else float(r)

            rows.append({
                "X_group": x_name,
                "Z_group": z_name,
                "n": n,
                "r1": r1,
                "p_perm": p,
                "status": "OK",
                "covars": "BMI+Sex_01"
            })

        except Exception as e:
            rows.append({
                "X_group": x_name, "Z_group": z_name,
                "n": n, "r1": np.nan, "p_perm": np.nan,
                "status": f"FAIL: {type(e).__name__}",
                "covars": "BMI+Sex_01"
            })

res_df_adj = pd.DataFrame(rows)

# 보기 좋게 정렬 (OK만 위로 + p_perm 오름차순)
res_df_sorted1_adj = (
    res_df_adj
    .assign(status_rank=lambda d: (d["status"] != "OK").astype(int))
    .sort_values(["status_rank", "p_perm"], ascending=[True, True])
    .drop(columns=["status_rank"])
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

res_df_sorted1_adj


,X_group,Z_group,n,r1,p_perm,status,covars
16,Liver,ConvT,12,0.900481,0.001998,OK,BMI+Sex_01
28,Kidney,B,22,0.606126,0.019980,OK,BMI+Sex_01
54,ETC,Detectin,18,0.624277,0.022977,OK,BMI+Sex_01
45,ETC,PAN,18,0.755322,0.039960,OK,BMI+Sex_01
50,ETC,GD1T,11,0.778242,0.044955,OK,BMI+Sex_01
31,Kidney,PAN,22,0.610306,0.045954,OK,BMI+Sex_01
14,Liver,B,21,0.671605,0.097902,OK,BMI+Sex_01
53,ETC,CD11c,18,0.627933,0.098901,OK,BMI+Sex_01
55,ETC,CD11c_1,18,0.565778,0.102897,OK,BMI+Sex_01
22,Liver,GD1T,12,0.789781,0.107892,OK,BMI+Sex_01


In [21]:
import statsmodels.api as sm
# df2통제후
# -----------------------------
# 0) residualize helper
# -----------------------------
def residualize_df(mat_df: pd.DataFrame, cov_df: pd.DataFrame) -> pd.DataFrame:
    """
    mat_df: (n x p) matrix dataframe (X or Z)
    cov_df: (n x k) covariates dataframe (BMI, Sex_01 etc.)
    return: residualized mat_df (same shape/cols)
    """
    Xc = sm.add_constant(cov_df.astype(float), has_constant="add")
    resid = {}
    for col in mat_df.columns:
        y = mat_df[col].astype(float).values
        fit = sm.OLS(y, Xc).fit()
        resid[col] = fit.resid
    return pd.DataFrame(resid, index=mat_df.index)

# -----------------------------
# 1) 그룹 정의 (원본 그대로)
# -----------------------------

X_groups = {
    "Liver":   ['AST', 'ALT', 'ALP', 'Bilirubin', 'Albumin', 'PT', 'APTT'],
    "Liver1" : ['AST', 'ALT', 'ALP', 'PT', 'APTT', 'Bilirubin'] , 
    "Liver2":["Albumin"],
    "Metabolic": ['TG', 'LDL',  'F_Glucose', 'HbA1c', 'Cholesterol','HDL'],
    "HDL": ['HDL'],
    "Metaboilic1":['TG', 'LDL',  'F_Glucose', 'HbA1c', 'Cholesterol'],
    "Kidney": ['BUN', 'Cr', 'GFR(CKD_EPI)', 'CrCl'],
    "Kidney1": ['BUN', 'Cr'],
    "Kidney2": ['GFR(CKD_EPI)', 'CrCl'],
}


Z_groups = {
    "B" : ["Transitional B", "Memory B", "Early Plasma B ", "Plasma B", "ABC"],
    "T" : ['NKT2', 'MAIT', 'gd1T','gd2T', 'ConvT%'],
    "ConvT": ['CD8+T', 'CD4+T', 'CD8+Treg', 'CD4+Treg'],
    "PAN": ['B', 'CD11b', 'HLADR', 'NK', 'NKT1', 'T'],
    "CD8":  ['CD8+T', 'CD8trm', 'CD8grzk'],
    "CD4":  ['CD4+T',  'CD4trm', 'CD4grzk'],
    "MAIT": ['MAIT', 'MAITtrm', 'MAITgrzK'],
    "CD20T":["CD20T"],
    "GD1T": ['gd1T','GD1Ttrm', 'GD1TgrzK'],
    "GD2T": ['gd2T','GD2Ttrm', 'GD2TgrzK'],
    "PCA" : ['CD8_PC1','CD4_PC1', 'MAIT_PC1', 'GD1T_PC1', 'GD2T_PC1'],
    "CD11c": ['CD11c+CD206-', 'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-',
              'Dectin-1+1','Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4'],
    "Detectin":[ 'Dectin-1+1','Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4'],
    "CD11c_1" :['CD11c+CD206-', 'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-']
}

# -----------------------------
# 2) covariates 지정
# -----------------------------
COVARS = ["BMI", 'Sex_01']   # <- 요청한 컬럼명

# -----------------------------
# 3) 실행 (BMI, Sex_01 통제한 sCCA)
# -----------------------------
rows = []
for x_name, X_cols in X_groups.items():
    for z_name, Z_cols in Z_groups.items():

        use_cols = X_cols + Z_cols + COVARS

        # 결측치 있는 행 제거 (covariate까지 포함해서 complete-case)
        df_clean = df2.dropna(subset=use_cols).copy()
        n = df_clean.shape[0]

        if n < 6:
            rows.append({
                "X_group": x_name, "Z_group": z_name,
                "n": n, "r1": np.nan, "p_perm": np.nan,
                "status": "SKIP(n too small)"
            })
            continue

        # 원본 행렬
        X_raw = df_clean[X_cols]
        Z_raw = df_clean[Z_cols]
        cov = df_clean[COVARS].astype(float)

        # ✅ 통제: residualize
        X = residualize_df(X_raw, cov)
        Z = residualize_df(Z_raw, cov)

        try:
            results = apply_sparse_cca(
                X, Z,
                X_cols=X_cols, Z_cols=Z_cols,
                verbose=False,
                visualize=False
            )

            r = results.get("correlations", None)
            p = results.get("permutation_pvalue", None)
            r1 = float(r[0]) if isinstance(r, (list, tuple, np.ndarray)) else float(r)

            rows.append({
                "X_group": x_name,
                "Z_group": z_name,
                "n": n,
                "r1": r1,
                "p_perm": p,
                "status": "OK",
                "covars": "BMI+Sex_01"
            })

        except Exception as e:
            rows.append({
                "X_group": x_name, "Z_group": z_name,
                "n": n, "r1": np.nan, "p_perm": np.nan,
                "status": f"FAIL: {type(e).__name__}",
                "covars": "BMI+Sex_01"
            })

res_df_adj = pd.DataFrame(rows)

# 보기 좋게 정렬 (OK만 위로 + p_perm 오름차순)
res_df_sorted1_adj = (
    res_df_adj
    .assign(status_rank=lambda d: (d["status"] != "OK").astype(int))
    .sort_values(["status_rank", "p_perm"], ascending=[True, True])
    .drop(columns=["status_rank"])
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

res_df_sorted1_adj


,X_group,Z_group,n,r1,p_perm,status,covars
37,Liver2,GD2T,22,0.739573,0.000999,OK,BMI+Sex_01
26,Liver1,Detectin,22,0.698093,0.002997,OK,BMI+Sex_01
9,Liver,GD2T,22,0.772969,0.003996,OK,BMI+Sex_01
59,HDL,PAN,22,0.716883,0.004995,OK,BMI+Sex_01
16,Liver1,ConvT,22,0.752618,0.007992,OK,BMI+Sex_01
2,Liver,ConvT,22,0.760004,0.013986,OK,BMI+Sex_01
53,Metabolic,CD11c,22,0.699935,0.020979,OK,BMI+Sex_01
12,Liver,Detectin,22,0.672616,0.028971,OK,BMI+Sex_01
118,Kidney2,MAIT,22,0.528346,0.049950,OK,BMI+Sex_01
15,Liver1,T,22,0.566483,0.053946,OK,BMI+Sex_01


### Task2 - Tsubset에 대해 plsr 돌리기 , 1:다

In [ ]:
df.columns

Index(['ID', 'Sex', 'Age', 'BMI', 'AST', 'ALT', 'ALP', 'Bilirubin', 'Protein',
       'Albumin', 'PT', 'APTT', 'Cholesterol', 'TG', 'LDL', 'HDL', 'R_Glucose',
       'F_Glucose', 'HbA1c', 'BUN', 'Cr', 'GFR(CKD_EPI)', 'CrCl', 'Amylase',
       'CRP', 'B', 'CD11b', 'HLADR', 'NK', 'NKT1', 'T', 'NKT2', 'MAIT', 'gd1T',
       'gd2T', 'ConvT%', 'CD8+T', 'CD4+T', 'CD8+Treg', 'CD4+Treg', 'CD20T',
       'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm', 'CD4grzk',
       'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK', 'Transitional B',
       'Memory B', 'Early Plasma B ', 'Plasma B', 'ABC', 'CD11c+CD206-',
       'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-', 'Dectin-1+1',
       'Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4', 'Gulucose', 'Sex_01',
       'CD8_PC1', 'CD4_PC1', 'MAIT_PC1', 'GD1T_PC1', 'GD2T_PC1', 'MAIT2_PC1',
       'MAIT_1_PC1', 'GD2T_1_PC1'],
      dtype='object')

#### 간 & df1 plsr

In [ ]:
sep = "_" * 90

X_cols = ['AST', 'ALT', 'ALP', 'Bilirubin', 'Protein', 'Albumin']

y_candidates = [
    'CD8_PC1', 'CD4_PC1', 'MAIT_PC1', 'GD1T_PC1', 'GD2T_PC1', 'MAIT_1_PC1',
    'GD2T_1_PC1', 'CD8_1_PC1', 'CD8_2_PC1', 'CD8_3_PC1', 'CD4_1_PC1',
    'CD4_2_PC1', 'CD4_3_PC1', 'MAIT_2_PC1', 'MAIT_3_PC1', 'GD1T_1_PC1',
    'GD1T_2_PC1', 'GD1T_3_PC1', 'GD2T_2_PC1', 'GD2T_3_PC1',
    'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환"""
    n = len(y)
    n_splits = min(n_splits, n)  # 안전
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def perm_pvalue_plsr_cv(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5,
                        n_perm=1000, random_state=42, statistic="abs_r"):
    """
    permutation p-value (CV 포함) 계산
    statistic:
      - "abs_r" : |corr(y, yhat_cv)|
      - "r2"    : corr(y, yhat_cv)^2
    """
    y = y.astype(float).to_numpy()
    y_pred = cv_pred_plsr(X, y, n_components=n_components, n_splits=n_splits, random_state=random_state)
    if y_pred is None:
        return np.nan, np.nan

    # 관측 통계량
    if np.std(y) < 1e-12 or np.std(y_pred) < 1e-12:
        r_obs = np.nan
        stat_obs = np.nan
    else:
        r_obs, _ = pearsonr(y, y_pred)
        stat_obs = abs(r_obs) if statistic == "abs_r" else (r_obs ** 2)

    if not np.isfinite(stat_obs):
        return r_obs, np.nan

    rng = np.random.default_rng(random_state)
    ge = 0

    # permutation: y를 셔플해서 CV를 다시 수행
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        y_perm_pred = cv_pred_plsr(X, y_perm, n_components=n_components, n_splits=n_splits, random_state=random_state)
        if y_perm_pred is None or np.std(y_perm_pred) < 1e-12:
            stat_perm = 0.0
        else:
            r_perm, _ = pearsonr(y_perm, y_perm_pred)
            stat_perm = abs(r_perm) if statistic == "abs_r" else (r_perm ** 2)

        if stat_perm >= stat_obs:
            ge += 1

    p_perm = (ge + 1) / (n_perm + 1)
    return r_obs, p_perm


def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    ※ PLS weight는 부호가 뒤집혀도 동등할 수 있어서, fold weight는 full-fit weight에 맞춰 부호 정렬합니다.
    """
    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    # x_weights_: (p, n_components), coef_: (p, 1)
    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    # 기준 방향(컴포넌트1만 정렬 기준으로 씀)
    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()  # component1

        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)  # (n_splits, p)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df1.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_obs": np.nan, "p_perm": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col]

    try:
        r_obs, p_perm = perm_pvalue_plsr_cv(
            X, y,
            n_components=1,
            n_splits=5,
            n_perm=1000,
            random_state=42,
            statistic="abs_r"  
        )

        print(f"obs_r = {r_obs:.4f}" if np.isfinite(r_obs) else "obs_r = NaN")
        print(f"perm_p = {p_perm:.4f}" if np.isfinite(p_perm) else "perm_p = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(
            X, y,
            n_components=1,
            n_splits=5,
            random_state=42
        )

        # 보기 좋게 합치기
        w_tbl = full_tbl.join(cv_tbl, how="left")

        # coef 절대값 기준 정렬 (기여 큰 순)
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_obs": r_obs, "p_perm": p_perm, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_obs": np.nan, "p_perm": np.nan, "status": f"FAIL: {type(e).__name__}"})

# 마지막 요약: permutation p-value 기준 정렬
res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("p_perm", ascending=True, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by permutation p-value)")
print(sep)
print(res_ok[["y_col", "n", "r_obs", "p_perm"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/30] TSubset = CD8_PC1
n = 12
obs_r = -0.5334
perm_p = 0.1548
------------------------------------------------------------
PLSR Weights / Coefs
           x_weight_c1    coef  cv_w_mean  cv_w_std
Albumin         0.6412 -0.5286     0.5494    0.1414
Protein         0.4752 -0.2945     0.3231    0.4123
ALT            -0.5644  0.0218    -0.5107    0.1564
Bilirubin      -0.0157  0.0200     0.0299    0.2401
ALP            -0.2037  0.0031    -0.1975    0.2612
AST             0.0521 -0.0022     0.0495    0.1514
------------------------------------------------------------
__________________________________________________________________________________________
[2/30] TSubset = CD4_PC1
n = 12
obs_r = -0.7388
perm_p = 0.0300
------------------------------------------------------------
PLSR Weights / Coefs
           x_weight_c1    coef  cv_w_mean  cv_w_std
Albumin         0.5539 -0.2892     0.4255    0.21

#### 신장

In [150]:
sep = "_" * 90

X_cols = ['BUN', 'Cr']
y_candidates = [
    'CD8_PC1', 'CD4_PC1', 'MAIT_PC1', 'GD1T_PC1', 'GD2T_PC1', 'MAIT_1_PC1',
    'GD2T_1_PC1', 'CD8_1_PC1', 'CD8_2_PC1', 'CD8_3_PC1', 'CD4_1_PC1',
    'CD4_2_PC1', 'CD4_3_PC1', 'MAIT_2_PC1', 'MAIT_3_PC1', 'GD1T_1_PC1',
    'GD1T_2_PC1', 'GD1T_3_PC1', 'GD2T_2_PC1', 'GD2T_3_PC1',
    'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환"""
    n = len(y)
    n_splits = min(n_splits, n)  # 안전
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def perm_pvalue_plsr_cv(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5,
                        n_perm=1000, random_state=42, statistic="abs_r"):
    """
    permutation p-value (CV 포함) 계산
    statistic:
      - "abs_r" : |corr(y, yhat_cv)|
      - "r2"    : corr(y, yhat_cv)^2
    """
    y = y.astype(float).to_numpy()
    y_pred = cv_pred_plsr(X, y, n_components=n_components, n_splits=n_splits, random_state=random_state)
    if y_pred is None:
        return np.nan, np.nan

    # 관측 통계량
    if np.std(y) < 1e-12 or np.std(y_pred) < 1e-12:
        r_obs = np.nan
        stat_obs = np.nan
    else:
        r_obs, _ = pearsonr(y, y_pred)
        stat_obs = abs(r_obs) if statistic == "abs_r" else (r_obs ** 2)

    if not np.isfinite(stat_obs):
        return r_obs, np.nan

    rng = np.random.default_rng(random_state)
    ge = 0

    # permutation: y를 셔플해서 CV를 다시 수행
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        y_perm_pred = cv_pred_plsr(X, y_perm, n_components=n_components, n_splits=n_splits, random_state=random_state)
        if y_perm_pred is None or np.std(y_perm_pred) < 1e-12:
            stat_perm = 0.0
        else:
            r_perm, _ = pearsonr(y_perm, y_perm_pred)
            stat_perm = abs(r_perm) if statistic == "abs_r" else (r_perm ** 2)

        if stat_perm >= stat_obs:
            ge += 1

    p_perm = (ge + 1) / (n_perm + 1)
    return r_obs, p_perm


def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    ※ PLS weight는 부호가 뒤집혀도 동등할 수 있어서, fold weight는 full-fit weight에 맞춰 부호 정렬합니다.
    """
    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    # x_weights_: (p, n_components), coef_: (p, 1)
    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    # 기준 방향(컴포넌트1만 정렬 기준으로 씀)
    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()  # component1

        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)  # (n_splits, p)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df1.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_obs": np.nan, "p_perm": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col]

    try:
        r_obs, p_perm = perm_pvalue_plsr_cv(
            X, y,
            n_components=1,
            n_splits=5,
            n_perm=1000,
            random_state=42,
            statistic="abs_r"  
        )

        print(f"obs_r = {r_obs:.4f}" if np.isfinite(r_obs) else "obs_r = NaN")
        print(f"perm_p = {p_perm:.4f}" if np.isfinite(p_perm) else "perm_p = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(
            X, y,
            n_components=1,
            n_splits=5,
            random_state=42
        )

        # 보기 좋게 합치기
        w_tbl = full_tbl.join(cv_tbl, how="left")

        # coef 절대값 기준 정렬 (기여 큰 순)
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_obs": r_obs, "p_perm": p_perm, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_obs": np.nan, "p_perm": np.nan, "status": f"FAIL: {type(e).__name__}"})

# 마지막 요약: permutation p-value 기준 정렬
res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("p_perm", ascending=True, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by permutation p-value)")
print(sep)
print(res_ok[["y_col", "n", "r_obs", "p_perm"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/30] TSubset = CD8_PC1
n = 13
obs_r = 0.2430
perm_p = 0.6234
------------------------------------------------------------
PLSR Weights / Coefs
     x_weight_c1    coef  cv_w_mean  cv_w_std
Cr       -0.6684 -2.5860    -0.6673    0.2791
BUN       0.7438  0.2456     0.6754    0.2127
------------------------------------------------------------
__________________________________________________________________________________________
[2/30] TSubset = CD4_PC1
n = 13
obs_r = -0.2293
perm_p = 0.6154
------------------------------------------------------------
PLSR Weights / Coefs
     x_weight_c1    coef  cv_w_mean  cv_w_std
Cr        0.9934 -1.8004     0.9073    0.1080
BUN      -0.1143  0.0177    -0.0346    0.4558
------------------------------------------------------------
__________________________________________________________________________________________
[3/30] TSubset = MAIT_PC1
n = 13
obs_r

#### df2_간

In [ ]:
sep = "_" * 90

X_cols =  ['AST', 'ALT', 'ALP', 'Bilirubin', 'Albumin', 'PT', 'APTT']

y_candidates = [
    'CD8_PC1', 'CD4_PC1', 'MAIT_PC1', 'GD1T_PC1', 'GD2T_PC1', 'MAIT_1_PC1',
    'GD2T_1_PC1', 'CD8_1_PC1', 'CD8_2_PC1', 'CD8_3_PC1', 'CD4_1_PC1',
    'CD4_2_PC1', 'CD4_3_PC1', 'MAIT_2_PC1', 'MAIT_3_PC1', 'GD1T_1_PC1',
    'GD1T_2_PC1', 'GD1T_3_PC1', 'GD2T_2_PC1', 'GD2T_3_PC1',
    'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환"""
    n = len(y)
    n_splits = min(n_splits, n)  # 안전
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def perm_pvalue_plsr_cv(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5,
                        n_perm=1000, random_state=42, statistic="abs_r"):
    """
    permutation p-value (CV 포함) 계산
    statistic:
      - "abs_r" : |corr(y, yhat_cv)|
      - "r2"    : corr(y, yhat_cv)^2
    """
    y = y.astype(float).to_numpy()
    y_pred = cv_pred_plsr(X, y, n_components=n_components, n_splits=n_splits, random_state=random_state)
    if y_pred is None:
        return np.nan, np.nan

    # 관측 통계량
    if np.std(y) < 1e-12 or np.std(y_pred) < 1e-12:
        r_obs = np.nan
        stat_obs = np.nan
    else:
        r_obs, _ = pearsonr(y, y_pred)
        stat_obs = abs(r_obs) if statistic == "abs_r" else (r_obs ** 2)

    if not np.isfinite(stat_obs):
        return r_obs, np.nan

    rng = np.random.default_rng(random_state)
    ge = 0

    # permutation: y를 셔플해서 CV를 다시 수행
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        y_perm_pred = cv_pred_plsr(X, y_perm, n_components=n_components, n_splits=n_splits, random_state=random_state)
        if y_perm_pred is None or np.std(y_perm_pred) < 1e-12:
            stat_perm = 0.0
        else:
            r_perm, _ = pearsonr(y_perm, y_perm_pred)
            stat_perm = abs(r_perm) if statistic == "abs_r" else (r_perm ** 2)

        if stat_perm >= stat_obs:
            ge += 1

    p_perm = (ge + 1) / (n_perm + 1)
    return r_obs, p_perm


def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    ※ PLS weight는 부호가 뒤집혀도 동등할 수 있어서, fold weight는 full-fit weight에 맞춰 부호 정렬합니다.
    """
    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    # x_weights_: (p, n_components), coef_: (p, 1)
    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    # 기준 방향(컴포넌트1만 정렬 기준으로 씀)
    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()  # component1

        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)  # (n_splits, p)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df2.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_obs": np.nan, "p_perm": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col]

    try:
        r_obs, p_perm = perm_pvalue_plsr_cv(
            X, y,
            n_components=1,
            n_splits=5,
            n_perm=1000,
            random_state=42,
            statistic="abs_r"  
        )

        print(f"obs_r = {r_obs:.4f}" if np.isfinite(r_obs) else "obs_r = NaN")
        print(f"perm_p = {p_perm:.4f}" if np.isfinite(p_perm) else "perm_p = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(
            X, y,
            n_components=1,
            n_splits=5,
            random_state=42
        )

        # 보기 좋게 합치기
        w_tbl = full_tbl.join(cv_tbl, how="left")

        # coef 절대값 기준 정렬 (기여 큰 순)
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_obs": r_obs, "p_perm": p_perm, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_obs": np.nan, "p_perm": np.nan, "status": f"FAIL: {type(e).__name__}"})

# 마지막 요약: permutation p-value 기준 정렬
res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("p_perm", ascending=True, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by permutation p-value)")
print(sep)
print(res_ok[["y_col", "n", "r_obs", "p_perm"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/30] TSubset = CD8_PC1
n = 22
obs_r = 0.1599
perm_p = 0.6414
------------------------------------------------------------
PLSR Weights / Coefs
           x_weight_c1    coef  cv_w_mean  cv_w_std
Bilirubin       0.5061  0.7251     0.4681    0.1384
Albumin         0.2404  0.2508     0.2314    0.1334
APTT           -0.5031 -0.0634    -0.4708    0.0841
ALT             0.4105  0.0255     0.4158    0.1550
PT              0.5133  0.0169     0.4613    0.0881
AST             0.0301  0.0025     0.0470    0.2411
ALP            -0.0122 -0.0003    -0.0261    0.1117
------------------------------------------------------------
__________________________________________________________________________________________
[2/30] TSubset = CD4_PC1
n = 22
obs_r = 0.0589
perm_p = 0.8561
------------------------------------------------------------
PLSR Weights / Coefs
           x_weight_c1    coef  cv_w_mean  cv_w_std

#### df2+신장

In [ ]:
# df2 + 신장
sep = "_" * 90

X_cols =  ['BUN', 'Cr', 'GFR(CKD_EPI)', 'CrCl']

y_candidates = [
    'CD8_PC1', 'CD4_PC1', 'MAIT_PC1', 'GD1T_PC1', 'GD2T_PC1', 'MAIT_1_PC1',
    'GD2T_1_PC1', 'CD8_1_PC1', 'CD8_2_PC1', 'CD8_3_PC1', 'CD4_1_PC1',
    'CD4_2_PC1', 'CD4_3_PC1', 'MAIT_2_PC1', 'MAIT_3_PC1', 'GD1T_1_PC1',
    'GD1T_2_PC1', 'GD1T_3_PC1', 'GD2T_2_PC1', 'GD2T_3_PC1',
    'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환"""
    n = len(y)
    n_splits = min(n_splits, n)  # 안전
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def perm_pvalue_plsr_cv(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5,
                        n_perm=1000, random_state=42, statistic="abs_r"):
    """
    permutation p-value (CV 포함) 계산
    statistic:
      - "abs_r" : |corr(y, yhat_cv)|
      - "r2"    : corr(y, yhat_cv)^2
    """
    y = y.astype(float).to_numpy()
    y_pred = cv_pred_plsr(X, y, n_components=n_components, n_splits=n_splits, random_state=random_state)
    if y_pred is None:
        return np.nan, np.nan

    # 관측 통계량
    if np.std(y) < 1e-12 or np.std(y_pred) < 1e-12:
        r_obs = np.nan
        stat_obs = np.nan
    else:
        r_obs, _ = pearsonr(y, y_pred)
        stat_obs = abs(r_obs) if statistic == "abs_r" else (r_obs ** 2)

    if not np.isfinite(stat_obs):
        return r_obs, np.nan

    rng = np.random.default_rng(random_state)
    ge = 0

    # permutation: y를 셔플해서 CV를 다시 수행
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        y_perm_pred = cv_pred_plsr(X, y_perm, n_components=n_components, n_splits=n_splits, random_state=random_state)
        if y_perm_pred is None or np.std(y_perm_pred) < 1e-12:
            stat_perm = 0.0
        else:
            r_perm, _ = pearsonr(y_perm, y_perm_pred)
            stat_perm = abs(r_perm) if statistic == "abs_r" else (r_perm ** 2)

        if stat_perm >= stat_obs:
            ge += 1

    p_perm = (ge + 1) / (n_perm + 1)
    return r_obs, p_perm


def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    ※ PLS weight는 부호가 뒤집혀도 동등할 수 있어서, fold weight는 full-fit weight에 맞춰 부호 정렬합니다.
    """
    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    # x_weights_: (p, n_components), coef_: (p, 1)
    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    # 기준 방향(컴포넌트1만 정렬 기준으로 씀)
    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()  # component1

        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)  # (n_splits, p)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df2.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_obs": np.nan, "p_perm": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col]

    try:
        r_obs, p_perm = perm_pvalue_plsr_cv(
            X, y,
            n_components=1,
            n_splits=5,
            n_perm=1000,
            random_state=42,
            statistic="abs_r"  
        )

        print(f"obs_r = {r_obs:.4f}" if np.isfinite(r_obs) else "obs_r = NaN")
        print(f"perm_p = {p_perm:.4f}" if np.isfinite(p_perm) else "perm_p = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(
            X, y,
            n_components=1,
            n_splits=5,
            random_state=42
        )

        # 보기 좋게 합치기
        w_tbl = full_tbl.join(cv_tbl, how="left")

        # coef 절대값 기준 정렬 (기여 큰 순)
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_obs": r_obs, "p_perm": p_perm, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_obs": np.nan, "p_perm": np.nan, "status": f"FAIL: {type(e).__name__}"})

# 마지막 요약: permutation p-value 기준 정렬
res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("p_perm", ascending=True, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by permutation p-value)")
print(sep)
print(res_ok[["y_col", "n", "r_obs", "p_perm"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/30] TSubset = CD8_PC1
n = 22
obs_r = -0.1455
perm_p = 0.6803
------------------------------------------------------------
PLSR Weights / Coefs
              x_weight_c1    coef  cv_w_mean  cv_w_std
Cr                 0.0711  0.2037     0.0264    0.3688
BUN               -0.3862 -0.0346    -0.4001    0.2409
GFR(CKD_EPI)      -0.6156 -0.0206    -0.5690    0.1496
CrCl               0.6832  0.0121     0.5479    0.2293
------------------------------------------------------------
__________________________________________________________________________________________
[2/30] TSubset = CD4_PC1
n = 22
obs_r = -0.0144
perm_p = 0.9650
------------------------------------------------------------
PLSR Weights / Coefs
              x_weight_c1    coef  cv_w_mean  cv_w_std
BUN                0.5309  0.0383     0.5216    0.2361
Cr                 0.0069  0.0160     0.0426    0.1119
GFR(CKD_EPI)      -0.5001

### 다시 해볼거 - trm grzk 묶은 조합만 남기고 + CV


In [72]:
sep = "_" * 90

X_cols = ['AST', 'ALT', 'ALP', 'Bilirubin', 'Protein', 'Albumin']

y_candidates = [
    'MAIT_1_PC1','GD2T_1_PC1', 'CD8_1_PC1', 'GD1T_1_PC1','CD4_1_PC1',
    'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환"""
    n = len(y)
    n_splits = min(n_splits, n)  # 안전
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def perm_pvalue_plsr_cv(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5,
                        n_perm=1000, random_state=42, statistic="abs_r"):
    """
    permutation p-value (CV 포함) 계산
    statistic:
      - "abs_r" : |corr(y, yhat_cv)|
      - "r2"    : corr(y, yhat_cv)^2
    """
    y = y.astype(float).to_numpy()
    y_pred = cv_pred_plsr(X, y, n_components=n_components, n_splits=n_splits, random_state=random_state)
    if y_pred is None:
        return np.nan, np.nan

    # 관측 통계량
    if np.std(y) < 1e-12 or np.std(y_pred) < 1e-12:
        r_obs = np.nan
        stat_obs = np.nan
    else:
        r_obs, _ = pearsonr(y, y_pred)
        stat_obs = abs(r_obs) if statistic == "abs_r" else (r_obs ** 2)

    if not np.isfinite(stat_obs):
        return r_obs, np.nan

    rng = np.random.default_rng(random_state)
    ge = 0

    # permutation: y를 셔플해서 CV를 다시 수행
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        y_perm_pred = cv_pred_plsr(X, y_perm, n_components=n_components, n_splits=n_splits, random_state=random_state)
        if y_perm_pred is None or np.std(y_perm_pred) < 1e-12:
            stat_perm = 0.0
        else:
            r_perm, _ = pearsonr(y_perm, y_perm_pred)
            stat_perm = abs(r_perm) if statistic == "abs_r" else (r_perm ** 2)

        if stat_perm >= stat_obs:
            ge += 1

    p_perm = (ge + 1) / (n_perm + 1)
    return r_obs, p_perm


def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    ※ PLS weight는 부호가 뒤집혀도 동등할 수 있어서, fold weight는 full-fit weight에 맞춰 부호 정렬합니다.
    """
    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    # x_weights_: (p, n_components), coef_: (p, 1)
    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    # 기준 방향(컴포넌트1만 정렬 기준으로 씀)
    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()  # component1

        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)  # (n_splits, p)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df1.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_obs": np.nan, "p_perm": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col]

    try:
        r_obs, p_perm = perm_pvalue_plsr_cv(
            X, y,
            n_components=1,
            n_splits=5,
            n_perm=1000,
            random_state=42,
            statistic="abs_r"  
        )

        print(f"obs_r = {r_obs:.4f}" if np.isfinite(r_obs) else "obs_r = NaN")
        print(f"perm_p = {p_perm:.4f}" if np.isfinite(p_perm) else "perm_p = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(
            X, y,
            n_components=1,
            n_splits=5,
            random_state=42
        )

        # 보기 좋게 합치기
        w_tbl = full_tbl.join(cv_tbl, how="left")

        # coef 절대값 기준 정렬 (기여 큰 순)
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_obs": r_obs, "p_perm": p_perm, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_obs": np.nan, "p_perm": np.nan, "status": f"FAIL: {type(e).__name__}"})

# 마지막 요약: permutation p-value 기준 정렬
res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("p_perm", ascending=True, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by permutation p-value)")
print(sep)
print(res_ok[["y_col", "n", "r_obs", "p_perm"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/15] TSubset = MAIT_1_PC1
n = 12
obs_r = -0.8470
perm_p = 0.0030
------------------------------------------------------------
PLSR Weights / Coefs
           x_weight_c1    coef  cv_w_mean  cv_w_std
Albumin         0.6533  0.2659     0.4946    0.2250
Protein         0.6918  0.2117     0.6519    0.0585
Bilirubin       0.0207  0.0131    -0.0164    0.1655
ALT            -0.2355 -0.0045    -0.0585    0.3227
AST             0.1256  0.0026     0.1834    0.1274
ALP             0.1512  0.0011     0.2591    0.2841
------------------------------------------------------------
__________________________________________________________________________________________
[2/15] TSubset = GD2T_1_PC1
n = 12
obs_r = -0.3416
perm_p = 0.4565
------------------------------------------------------------
PLSR Weights / Coefs
           x_weight_c1    coef  cv_w_mean  cv_w_std
Protein         0.6297 -0.5425     0.5024  

In [93]:
sep = "_" * 90

X_cols = ['BUN', 'Cr']
y_candidates = [
    'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환"""
    n = len(y)
    n_splits = min(n_splits, n)  # 안전
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def perm_pvalue_plsr_cv(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5,
                        n_perm=1000, random_state=42, statistic="abs_r"):
    """
    permutation p-value (CV 포함) 계산
    statistic:
      - "abs_r" : |corr(y, yhat_cv)|
      - "r2"    : corr(y, yhat_cv)^2
    """
    y = y.astype(float).to_numpy()
    y_pred = cv_pred_plsr(X, y, n_components=n_components, n_splits=n_splits, random_state=random_state)
    if y_pred is None:
        return np.nan, np.nan

    # 관측 통계량
    if np.std(y) < 1e-12 or np.std(y_pred) < 1e-12:
        r_obs = np.nan
        stat_obs = np.nan
    else:
        r_obs, _ = pearsonr(y, y_pred)
        stat_obs = abs(r_obs) if statistic == "abs_r" else (r_obs ** 2)

    if not np.isfinite(stat_obs):
        return r_obs, np.nan

    rng = np.random.default_rng(random_state)
    ge = 0

    # permutation: y를 셔플해서 CV를 다시 수행
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        y_perm_pred = cv_pred_plsr(X, y_perm, n_components=n_components, n_splits=n_splits, random_state=random_state)
        if y_perm_pred is None or np.std(y_perm_pred) < 1e-12:
            stat_perm = 0.0
        else:
            r_perm, _ = pearsonr(y_perm, y_perm_pred)
            stat_perm = abs(r_perm) if statistic == "abs_r" else (r_perm ** 2)

        if stat_perm >= stat_obs:
            ge += 1

    p_perm = (ge + 1) / (n_perm + 1)
    return r_obs, p_perm


def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    ※ PLS weight는 부호가 뒤집혀도 동등할 수 있어서, fold weight는 full-fit weight에 맞춰 부호 정렬합니다.
    """
    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    # x_weights_: (p, n_components), coef_: (p, 1)
    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    # 기준 방향(컴포넌트1만 정렬 기준으로 씀)
    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()  # component1

        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)  # (n_splits, p)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df1.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_obs": np.nan, "p_perm": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col]

    try:
        r_obs, p_perm = perm_pvalue_plsr_cv(
            X, y,
            n_components=1,
            n_splits=5,
            n_perm=1000,
            random_state=42,
            statistic="abs_r"  
        )

        print(f"obs_r = {r_obs:.4f}" if np.isfinite(r_obs) else "obs_r = NaN")
        print(f"perm_p = {p_perm:.4f}" if np.isfinite(p_perm) else "perm_p = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(
            X, y,
            n_components=1,
            n_splits=5,
            random_state=42
        )

        # 보기 좋게 합치기
        w_tbl = full_tbl.join(cv_tbl, how="left")

        # coef 절대값 기준 정렬 (기여 큰 순)
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_obs": r_obs, "p_perm": p_perm, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_obs": np.nan, "p_perm": np.nan, "status": f"FAIL: {type(e).__name__}"})

# 마지막 요약: permutation p-value 기준 정렬
res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("p_perm", ascending=True, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by permutation p-value)")
print(sep)
print(res_ok[["y_col", "n", "r_obs", "p_perm"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/10] TSubset = CD4trm
n = 13
obs_r = -0.0588
perm_p = 0.9011
------------------------------------------------------------
PLSR Weights / Coefs
     x_weight_c1     coef  cv_w_mean  cv_w_std
Cr         0.895 -30.6042     0.8487    0.1818
BUN       -0.446   1.3016    -0.4460    0.2605
------------------------------------------------------------
__________________________________________________________________________________________
[2/10] TSubset = CD8trm
n = 13
obs_r = -0.1508
perm_p = 0.7562
------------------------------------------------------------
PLSR Weights / Coefs
     x_weight_c1     coef  cv_w_mean  cv_w_std
Cr        0.9579 -31.2569     0.9554    0.0223
BUN      -0.2870   0.7991    -0.2868    0.0755
------------------------------------------------------------
__________________________________________________________________________________________
[3/10] TSubset = MAITtrm
n = 13


KeyboardInterrupt: 

In [78]:
df.columns

Index(['ID', 'Sex', 'Age', 'BMI', 'AST', 'ALT', 'ALP', 'Bilirubin', 'Protein',
       'Albumin', 'PT', 'APTT', 'Cholesterol', 'TG', 'LDL', 'HDL', 'R_Glucose',
       'F_Glucose', 'HbA1c', 'BUN', 'Cr', 'GFR(CKD_EPI)', 'CrCl', 'Amylase',
       'CRP', 'B', 'CD11b', 'HLADR', 'NK', 'NKT1', 'T', 'NKT2', 'MAIT', 'gd1T',
       'gd2T', 'ConvT%', 'CD8+T', 'CD4+T', 'CD8+Treg', 'CD4+Treg', 'CD20T',
       'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm', 'CD4grzk',
       'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK', 'Transitional B',
       'Memory B', 'Early Plasma B ', 'Plasma B', 'ABC', 'CD11c+CD206-',
       'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-', 'Dectin-1+1',
       'Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4', 'Gulucose', 'Sex_01',
       'CD8_PC1', 'CD8_1_PC1', 'CD8_2_PC1', 'CD8_3_PC1', 'CD4_PC1',
       'CD4_1_PC1', 'CD4_2_PC1', 'CD4_3_PC1', 'MAIT_PC1', 'MAIT_1_PC1',
       'MAIT_2_PC1', 'MAIT_3_PC1', 'GD1T_PC1', 'GD1T_1_PC1', 'GD1T_2_PC1',
       'GD1T_3_PC1', 'GD2T_PC1', 'GD2

In [79]:
# df2 + 간 - 안나와야댐
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
from scipy.stats import pearsonr

sep = "_" * 90

# =========================
# df2 간(liver)
# =========================
X_cols = ['AST', 'ALT', 'ALP', 'Bilirubin', 'Albumin', 'PT', 'APTT']

y_candidates = [ 'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환(원래 순서대로)"""
    n = len(y)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def cv_metrics(y_true, y_pred):
    """r_cv, Q2, RMSECV 계산"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if (np.std(y_true) < 1e-12) or (np.std(y_pred) < 1e-12):
        r = np.nan
    else:
        r, _ = pearsonr(y_true, y_pred)

    sse = np.sum((y_true - y_pred) ** 2)
    sst = np.sum((y_true - np.mean(y_true)) ** 2)
    Q2 = np.nan if sst < 1e-12 else (1.0 - sse / sst)

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return r, Q2, rmse

def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    ※ PLS weight는 부호가 뒤집혀도 동등 → fold weight는 full-fit weight에 맞춰 부호 정렬
    """
    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()  # component1

        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행 (CV 기준)
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df2.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col].astype(float).to_numpy()

    try:
        y_pred = cv_pred_plsr(X, y, n_components=1, n_splits=5, random_state=42)
        if y_pred is None:
            raise ValueError("CV split too small")

        r_cv, Q2, rmse = cv_metrics(y, y_pred)

        print(f"r_cv   = {r_cv:.4f}" if np.isfinite(r_cv) else "r_cv   = NaN")
        print(f"Q2     = {Q2:.4f}" if np.isfinite(Q2) else "Q2     = NaN")
        print(f"RMSECV = {rmse:.4f}" if np.isfinite(rmse) else "RMSECV = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(
            X, pd.Series(y, index=df_xy.index),
            n_components=1,
            n_splits=5,
            random_state=42
        )

        w_tbl = full_tbl.join(cv_tbl, how="left")
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_cv": r_cv, "Q2": Q2, "RMSECV": rmse, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": f"FAIL: {type(e).__name__}"})


# 마지막 요약: CV Q2 기준 정렬(내림차순)
res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("Q2", ascending=False, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by CV Q2, desc)")
print(sep)
print(res_ok[["y_col", "n", "r_cv", "Q2", "RMSECV"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/10] TSubset = CD4trm
n = 22
r_cv   = 0.2493
Q2     = -0.0183
RMSECV = 15.8088
------------------------------------------------------------
PLSR Weights / Coefs
           x_weight_c1    coef  cv_w_mean  cv_w_std
Bilirubin       0.3585  9.2093     0.3433    0.1365
Albumin         0.1280  2.3950     0.1126    0.0689
APTT           -0.5931 -1.3407    -0.5558    0.0662
PT              0.6429  0.3793     0.6155    0.0616
AST             0.1596  0.2396     0.1349    0.2448
ALP            -0.2430 -0.0904    -0.2399    0.1239
ALT            -0.0745 -0.0831    -0.0956    0.1191
------------------------------------------------------------
__________________________________________________________________________________________
[2/10] TSubset = CD8trm
n = 22
r_cv   = 0.1120
Q2     = -0.0769
RMSECV = 19.0989
------------------------------------------------------------
PLSR Weights / Coefs
           x_we

In [91]:
df2.columns

Index(['ID', 'Sex', 'Age', 'BMI', 'AST', 'ALT', 'ALP', 'Bilirubin', 'Protein',
       'Albumin', 'PT', 'APTT', 'Cholesterol', 'TG', 'LDL', 'HDL', 'R_Glucose',
       'F_Glucose', 'HbA1c', 'BUN', 'Cr', 'GFR(CKD_EPI)', 'CrCl', 'Amylase',
       'CRP', 'B', 'CD11b', 'HLADR', 'NK', 'NKT1', 'T', 'NKT2', 'MAIT', 'gd1T',
       'gd2T', 'ConvT%', 'CD8+T', 'CD4+T', 'CD8+Treg', 'CD4+Treg', 'CD20T',
       'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm', 'CD4grzk',
       'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK', 'Transitional B',
       'Memory B', 'Early Plasma B ', 'Plasma B', 'ABC', 'CD11c+CD206-',
       'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-', 'Dectin-1+1',
       'Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4', 'Gulucose', 'Sex_01',
       'CD8_PC1', 'CD8_1_PC1', 'CD8_2_PC1', 'CD8_3_PC1', 'CD4_PC1',
       'CD4_1_PC1', 'CD4_2_PC1', 'CD4_3_PC1', 'MAIT_PC1', 'MAIT_1_PC1',
       'MAIT_2_PC1', 'MAIT_3_PC1', 'GD1T_PC1', 'GD1T_1_PC1', 'GD1T_2_PC1',
       'GD1T_3_PC1', 'GD2T_PC1', 'GD2

In [92]:
# df2 +신장 cv

import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
from scipy.stats import pearsonr

sep = "_" * 90

X_cols = [ 'BUN', 'Cr', 'GFR(CKD_EPI)', 'CrCl']

y_candidates = [ 'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]


def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환(원래 인덱스 순서대로)"""
    n = len(y)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def cv_metrics(y_true, y_pred):
    """r_cv, Q2, RMSECV 계산"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if (np.std(y_true) < 1e-12) or (np.std(y_pred) < 1e-12):
        r = np.nan
    else:
        r, _ = pearsonr(y_true, y_pred)

    sse = np.sum((y_true - y_pred) ** 2)
    sst = np.sum((y_true - np.mean(y_true)) ** 2)

    Q2 = np.nan if sst < 1e-12 else (1.0 - sse / sst)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    return r, Q2, rmse

def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    """
    import statsmodels.api as sm  # 혹시 안 쓰면 삭제해도 됨 (여기선 사용 안 함)

    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()

        # 부호 정렬
        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행 (CV 성능 기준)
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df2.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col].astype(float).to_numpy()

    try:
        y_pred = cv_pred_plsr(X, y, n_components=1, n_splits=5, random_state=42)
        if y_pred is None:
            raise ValueError("CV split too small")

        r_cv, Q2, rmse = cv_metrics(y, y_pred)

        print(f"r_cv   = {r_cv:.4f}" if np.isfinite(r_cv) else "r_cv   = NaN")
        print(f"Q2     = {Q2:.4f}" if np.isfinite(Q2) else "Q2     = NaN")
        print(f"RMSECV = {rmse:.4f}" if np.isfinite(rmse) else "RMSECV = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(X, pd.Series(y, index=df_xy.index), n_components=1, n_splits=5, random_state=42)
        w_tbl = full_tbl.join(cv_tbl, how="left")
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_cv": r_cv, "Q2": Q2, "RMSECV": rmse, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": f"FAIL: {type(e).__name__}"})


res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("Q2", ascending=False, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by CV Q2, desc)")
print(sep)
print(res_ok[["y_col", "n", "r_cv", "Q2", "RMSECV"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/10] TSubset = CD4trm
n = 22
r_cv   = -0.0490
Q2     = -0.1893
RMSECV = 17.0846
------------------------------------------------------------
PLSR Weights / Coefs
              x_weight_c1    coef  cv_w_mean  cv_w_std
Cr                -0.2283  7.1559    -0.2674    0.3379
GFR(CKD_EPI)       0.8199 -0.3006     0.7399    0.1168
BUN               -0.2327  0.2282    -0.1941    0.2451
CrCl              -0.4706  0.0913    -0.4075    0.1838
------------------------------------------------------------
__________________________________________________________________________________________
[2/10] TSubset = CD8trm
n = 22
r_cv   = -0.0097
Q2     = -0.2825
RMSECV = 20.8421
------------------------------------------------------------
PLSR Weights / Coefs
              x_weight_c1    coef  cv_w_mean  cv_w_std
Cr                -0.0558  2.4360    -0.0551    0.2650
GFR(CKD_EPI)       0.6919 -0.3534     0.6793

In [94]:
# df1 + 신장
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
from scipy.stats import pearsonr

sep = "_" * 90

X_cols = ['BUN', 'Cr']
y_candidates = [
    'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환(원래 인덱스 순서대로)"""
    n = len(y)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def cv_metrics(y_true, y_pred):
    """r_cv, Q2, RMSECV 계산"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if (np.std(y_true) < 1e-12) or (np.std(y_pred) < 1e-12):
        r = np.nan
    else:
        r, _ = pearsonr(y_true, y_pred)

    sse = np.sum((y_true - y_pred) ** 2)
    sst = np.sum((y_true - np.mean(y_true)) ** 2)

    Q2 = np.nan if sst < 1e-12 else (1.0 - sse / sst)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    return r, Q2, rmse

def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    """
    import statsmodels.api as sm  # 혹시 안 쓰면 삭제해도 됨 (여기선 사용 안 함)

    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()

        # 부호 정렬
        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행 (CV 성능 기준)
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df1.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col].astype(float).to_numpy()

    try:
        y_pred = cv_pred_plsr(X, y, n_components=1, n_splits=5, random_state=42)
        if y_pred is None:
            raise ValueError("CV split too small")

        r_cv, Q2, rmse = cv_metrics(y, y_pred)

        print(f"r_cv   = {r_cv:.4f}" if np.isfinite(r_cv) else "r_cv   = NaN")
        print(f"Q2     = {Q2:.4f}" if np.isfinite(Q2) else "Q2     = NaN")
        print(f"RMSECV = {rmse:.4f}" if np.isfinite(rmse) else "RMSECV = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(X, pd.Series(y, index=df_xy.index), n_components=1, n_splits=5, random_state=42)
        w_tbl = full_tbl.join(cv_tbl, how="left")
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_cv": r_cv, "Q2": Q2, "RMSECV": rmse, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": f"FAIL: {type(e).__name__}"})


res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("Q2", ascending=False, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by CV Q2, desc)")
print(sep)
print(res_ok[["y_col", "n", "r_cv", "Q2", "RMSECV"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/10] TSubset = CD4trm
n = 13
r_cv   = -0.0588
Q2     = -0.3979
RMSECV = 19.9140
------------------------------------------------------------
PLSR Weights / Coefs
     x_weight_c1     coef  cv_w_mean  cv_w_std
Cr         0.895 -30.6042     0.8487    0.1818
BUN       -0.446   1.3016    -0.4460    0.2605
------------------------------------------------------------
__________________________________________________________________________________________
[2/10] TSubset = CD8trm
n = 13
r_cv   = -0.1508
Q2     = -0.4136
RMSECV = 20.6309
------------------------------------------------------------
PLSR Weights / Coefs
     x_weight_c1     coef  cv_w_mean  cv_w_std
Cr        0.9579 -31.2569     0.9554    0.0223
BUN      -0.2870   0.7991    -0.2868    0.0755
------------------------------------------------------------
______________________________________________________________________________________

In [95]:
# df1 + 간
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
from scipy.stats import pearsonr

sep = "_" * 90

X_cols =  ['AST', 'ALT', 'ALP', 'Bilirubin', 'Protein', 'Albumin']
y_candidates = [
    'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환(원래 인덱스 순서대로)"""
    n = len(y)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def cv_metrics(y_true, y_pred):
    """r_cv, Q2, RMSECV 계산"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if (np.std(y_true) < 1e-12) or (np.std(y_pred) < 1e-12):
        r = np.nan
    else:
        r, _ = pearsonr(y_true, y_pred)

    sse = np.sum((y_true - y_pred) ** 2)
    sst = np.sum((y_true - np.mean(y_true)) ** 2)

    Q2 = np.nan if sst < 1e-12 else (1.0 - sse / sst)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    return r, Q2, rmse

def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    """
    import statsmodels.api as sm  # 혹시 안 쓰면 삭제해도 됨 (여기선 사용 안 함)

    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()

        # 부호 정렬
        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행 (CV 성능 기준)
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df1.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col].astype(float).to_numpy()

    try:
        y_pred = cv_pred_plsr(X, y, n_components=1, n_splits=5, random_state=42)
        if y_pred is None:
            raise ValueError("CV split too small")

        r_cv, Q2, rmse = cv_metrics(y, y_pred)

        print(f"r_cv   = {r_cv:.4f}" if np.isfinite(r_cv) else "r_cv   = NaN")
        print(f"Q2     = {Q2:.4f}" if np.isfinite(Q2) else "Q2     = NaN")
        print(f"RMSECV = {rmse:.4f}" if np.isfinite(rmse) else "RMSECV = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(X, pd.Series(y, index=df_xy.index), n_components=1, n_splits=5, random_state=42)
        w_tbl = full_tbl.join(cv_tbl, how="left")
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_cv": r_cv, "Q2": Q2, "RMSECV": rmse, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": f"FAIL: {type(e).__name__}"})


res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("Q2", ascending=False, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by CV Q2, desc)")
print(sep)
print(res_ok[["y_col", "n", "r_cv", "Q2", "RMSECV"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/10] TSubset = CD4trm
n = 12
r_cv   = -0.3849
Q2     = -0.4430
RMSECV = 20.9134
------------------------------------------------------------
PLSR Weights / Coefs
           x_weight_c1    coef  cv_w_mean  cv_w_std
Albumin         0.6381 -4.8556     0.5512    0.0705
Protein         0.4336 -2.4800     0.2627    0.3682
Bilirubin       0.1556 -1.8352     0.1289    0.2905
ALT            -0.6164  0.2196    -0.5349    0.2344
AST            -0.0105  0.0041    -0.0063    0.1555
ALP            -0.0232  0.0033     0.0108    0.3182
------------------------------------------------------------
__________________________________________________________________________________________
[2/10] TSubset = CD8trm
n = 12
r_cv   = -0.6169
Q2     = -1.3436
RMSECV = 27.6238
------------------------------------------------------------
PLSR Weights / Coefs
           x_weight_c1    coef  cv_w_mean  cv_w_std
Albumin      

### PLSR X cols나눠서 해보깅. . .

In [26]:

# df 2 1번간

# df2 + 간 - 안나와야댐
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
from scipy.stats import pearsonr

sep = "_" * 90

# =========================
# df2 간(liver)
# =========================
X_cols = ['AST', 'ALT', 'ALP', 'Bilirubin', 'PT', 'APTT']

y_candidates = [ 'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환(원래 순서대로)"""
    n = len(y)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def cv_metrics(y_true, y_pred):
    """r_cv, Q2, RMSECV 계산"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if (np.std(y_true) < 1e-12) or (np.std(y_pred) < 1e-12):
        r = np.nan
    else:
        r, _ = pearsonr(y_true, y_pred)

    sse = np.sum((y_true - y_pred) ** 2)
    sst = np.sum((y_true - np.mean(y_true)) ** 2)
    Q2 = np.nan if sst < 1e-12 else (1.0 - sse / sst)

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return r, Q2, rmse

def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    ※ PLS weight는 부호가 뒤집혀도 동등 → fold weight는 full-fit weight에 맞춰 부호 정렬
    """
    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()  # component1

        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행 (CV 기준)
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df2.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col].astype(float).to_numpy()

    try:
        y_pred = cv_pred_plsr(X, y, n_components=1, n_splits=5, random_state=42)
        if y_pred is None:
            raise ValueError("CV split too small")

        r_cv, Q2, rmse = cv_metrics(y, y_pred)

        print(f"r_cv   = {r_cv:.4f}" if np.isfinite(r_cv) else "r_cv   = NaN")
        print(f"Q2     = {Q2:.4f}" if np.isfinite(Q2) else "Q2     = NaN")
        print(f"RMSECV = {rmse:.4f}" if np.isfinite(rmse) else "RMSECV = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(
            X, pd.Series(y, index=df_xy.index),
            n_components=1,
            n_splits=5,
            random_state=42
        )

        w_tbl = full_tbl.join(cv_tbl, how="left")
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_cv": r_cv, "Q2": Q2, "RMSECV": rmse, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": f"FAIL: {type(e).__name__}"})


# 마지막 요약: CV Q2 기준 정렬(내림차순)
res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("Q2", ascending=False, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by CV Q2, desc)")
print(sep)
print(res_ok[["y_col", "n", "r_cv", "Q2", "RMSECV"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/10] TSubset = CD4trm
n = 22
r_cv   = 0.2585
Q2     = -0.0072
RMSECV = 15.7220
------------------------------------------------------------
PLSR Weights / Coefs
           x_weight_c1    coef  cv_w_mean  cv_w_std
Bilirubin       0.3615  9.5052     0.3464    0.1375
APTT           -0.5980 -1.3838    -0.5607    0.0689
PT              0.6482  0.3915     0.6207    0.0625
AST             0.1610  0.2473     0.1360    0.2460
ALP            -0.2450 -0.0933    -0.2418    0.1240
ALT            -0.0751 -0.0858    -0.0959    0.1192
------------------------------------------------------------
__________________________________________________________________________________________
[2/10] TSubset = CD8trm
n = 22
r_cv   = 0.0897
Q2     = -0.0822
RMSECV = 19.1455
------------------------------------------------------------
PLSR Weights / Coefs
           x_weight_c1     coef  cv_w_mean  cv_w_std
Bilirubin     

In [ ]:
# df2 + 간 - 안나와야댐 -2번 간
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
from scipy.stats import pearsonr

sep = "_" * 90

# =========================
# df2 간(liver)
# =========================
X_cols = ['Albumin']

y_candidates = [ 'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환(원래 순서대로)"""
    n = len(y)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def cv_metrics(y_true, y_pred):
    """r_cv, Q2, RMSECV 계산"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if (np.std(y_true) < 1e-12) or (np.std(y_pred) < 1e-12):
        r = np.nan
    else:
        r, _ = pearsonr(y_true, y_pred)

    sse = np.sum((y_true - y_pred) ** 2)
    sst = np.sum((y_true - np.mean(y_true)) ** 2)
    Q2 = np.nan if sst < 1e-12 else (1.0 - sse / sst)

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return r, Q2, rmse

def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    ※ PLS weight는 부호가 뒤집혀도 동등 → fold weight는 full-fit weight에 맞춰 부호 정렬
    """
    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()  # component1

        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행 (CV 기준)
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df2.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col].astype(float).to_numpy()

    try:
        y_pred = cv_pred_plsr(X, y, n_components=1, n_splits=5, random_state=42)
        if y_pred is None:
            raise ValueError("CV split too small")

        r_cv, Q2, rmse = cv_metrics(y, y_pred)

        print(f"r_cv   = {r_cv:.4f}" if np.isfinite(r_cv) else "r_cv   = NaN")
        print(f"Q2     = {Q2:.4f}" if np.isfinite(Q2) else "Q2     = NaN")
        print(f"RMSECV = {rmse:.4f}" if np.isfinite(rmse) else "RMSECV = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(
            X, pd.Series(y, index=df_xy.index),
            n_components=1,
            n_splits=5,
            random_state=42
        )

        w_tbl = full_tbl.join(cv_tbl, how="left")
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_cv": r_cv, "Q2": Q2, "RMSECV": rmse, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": f"FAIL: {type(e).__name__}"})


# 마지막 요약: CV Q2 기준 정렬(내림차순)
res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("Q2", ascending=False, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by CV Q2, desc)")
print(sep)
print(res_ok[["y_col", "n", "r_cv", "Q2", "RMSECV"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/10] TSubset = CD4trm
n = 22
r_cv   = 0.3648
Q2     = 0.1017
RMSECV = 14.8483
------------------------------------------------------------
PLSR Weights / Coefs
      x_weight_c1   coef  cv_w_mean  cv_w_std
APTT      -0.6781 -1.753    -0.6675    0.0731
PT         0.7350  0.496     0.7392    0.0677
------------------------------------------------------------
__________________________________________________________________________________________
[2/10] TSubset = CD8trm
n = 22
r_cv   = 0.1737
Q2     = -0.0244
RMSECV = 18.6269
------------------------------------------------------------
PLSR Weights / Coefs
      x_weight_c1    coef  cv_w_mean  cv_w_std
APTT      -0.6535 -1.4801    -0.6247    0.1028
PT         0.7569  0.4475     0.7707    0.0954
------------------------------------------------------------
__________________________________________________________________________________________
[

In [27]:
df1.columns

Index(['ID', 'Sex', 'Age', 'BMI', 'AST', 'ALT', 'ALP', 'Bilirubin', 'Protein',
       'Albumin', 'PT', 'APTT', 'Cholesterol', 'TG', 'LDL', 'HDL', 'R_Glucose',
       'F_Glucose', 'HbA1c', 'BUN', 'Cr', 'GFR(CKD_EPI)', 'CrCl', 'Amylase',
       'CRP', 'B', 'CD11b', 'HLADR', 'NK', 'NKT1', 'T', 'NKT2', 'MAIT', 'gd1T',
       'gd2T', 'ConvT%', 'CD8+T', 'CD4+T', 'CD8+Treg', 'CD4+Treg', 'CD20T',
       'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm', 'CD4grzk',
       'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK', 'Transitional B',
       'Memory B', 'Early Plasma B ', 'Plasma B', 'ABC', 'CD11c+CD206-',
       'CD11c+CD206+', 'CD11c-CD206+', 'CD11c-CD206-', 'Dectin-1+1',
       'Dectin-1+2', 'Dectin-1+3', 'Dectin-1+4', 'Gulucose', 'Sex_01',
       'CD8_PC1', 'CD8_1_PC1', 'CD8_2_PC1', 'CD8_3_PC1', 'CD4_PC1',
       'CD4_1_PC1', 'CD4_2_PC1', 'CD4_3_PC1', 'MAIT_PC1', 'MAIT_1_PC1',
       'MAIT_2_PC1', 'MAIT_3_PC1', 'GD1T_PC1', 'GD1T_1_PC1', 'GD1T_2_PC1',
       'GD1T_3_PC1', 'GD2T_PC1', 'GD2

In [28]:
# df1 + 간 1
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
from scipy.stats import pearsonr

sep = "_" * 90

# =========================
# df2 간(liver)
# =========================
X_cols = ['AST', 'ALT', 'ALP', 'Bilirubin']

y_candidates = [ 'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환(원래 순서대로)"""
    n = len(y)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def cv_metrics(y_true, y_pred):
    """r_cv, Q2, RMSECV 계산"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if (np.std(y_true) < 1e-12) or (np.std(y_pred) < 1e-12):
        r = np.nan
    else:
        r, _ = pearsonr(y_true, y_pred)

    sse = np.sum((y_true - y_pred) ** 2)
    sst = np.sum((y_true - np.mean(y_true)) ** 2)
    Q2 = np.nan if sst < 1e-12 else (1.0 - sse / sst)

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return r, Q2, rmse

def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    ※ PLS weight는 부호가 뒤집혀도 동등 → fold weight는 full-fit weight에 맞춰 부호 정렬
    """
    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()  # component1

        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행 (CV 기준)
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df1.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col].astype(float).to_numpy()

    try:
        y_pred = cv_pred_plsr(X, y, n_components=1, n_splits=5, random_state=42)
        if y_pred is None:
            raise ValueError("CV split too small")

        r_cv, Q2, rmse = cv_metrics(y, y_pred)

        print(f"r_cv   = {r_cv:.4f}" if np.isfinite(r_cv) else "r_cv   = NaN")
        print(f"Q2     = {Q2:.4f}" if np.isfinite(Q2) else "Q2     = NaN")
        print(f"RMSECV = {rmse:.4f}" if np.isfinite(rmse) else "RMSECV = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(
            X, pd.Series(y, index=df_xy.index),
            n_components=1,
            n_splits=5,
            random_state=42
        )

        w_tbl = full_tbl.join(cv_tbl, how="left")
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_cv": r_cv, "Q2": Q2, "RMSECV": rmse, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": f"FAIL: {type(e).__name__}"})


# 마지막 요약: CV Q2 기준 정렬(내림차순)
res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("Q2", ascending=False, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by CV Q2, desc)")
print(sep)
print(res_ok[["y_col", "n", "r_cv", "Q2", "RMSECV"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/10] TSubset = CD4trm
n = 12
r_cv   = -0.5621
Q2     = -0.7001
RMSECV = 22.7000
------------------------------------------------------------
PLSR Weights / Coefs
           x_weight_c1    coef  cv_w_mean  cv_w_std
Bilirubin      -0.2446 -3.6666    -0.1707    0.3729
ALT             0.9688  0.4387     0.7354    0.2968
AST             0.0164  0.0082     0.0034    0.2230
ALP             0.0364  0.0066    -0.0775    0.5031
------------------------------------------------------------
__________________________________________________________________________________________
[2/10] TSubset = CD8trm
n = 12
r_cv   = -0.6943
Q2     = -0.8002
RMSECV = 24.2103
------------------------------------------------------------
PLSR Weights / Coefs
           x_weight_c1    coef  cv_w_mean  cv_w_std
Bilirubin      -0.1858 -2.2732    -0.1290    0.3855
ALT             0.9710  0.3588     0.6655    0.2745
AST          

In [ ]:
# df1 + 간 2
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
from scipy.stats import pearsonr

sep = "_" * 90

# =========================
# df2 간(liver)
# =========================
X_cols = ["Protein", "Albumin"]

y_candidates = [ 'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환(원래 순서대로)"""
    n = len(y)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def cv_metrics(y_true, y_pred):
    """r_cv, Q2, RMSECV 계산"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if (np.std(y_true) < 1e-12) or (np.std(y_pred) < 1e-12):
        r = np.nan
    else:
        r, _ = pearsonr(y_true, y_pred)

    sse = np.sum((y_true - y_pred) ** 2)
    sst = np.sum((y_true - np.mean(y_true)) ** 2)
    Q2 = np.nan if sst < 1e-12 else (1.0 - sse / sst)

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return r, Q2, rmse

def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    ※ PLS weight는 부호가 뒤집혀도 동등 → fold weight는 full-fit weight에 맞춰 부호 정렬
    """
    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()  # component1

        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행 (CV 기준)
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df1.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col].astype(float).to_numpy()

    try:
        y_pred = cv_pred_plsr(X, y, n_components=1, n_splits=5, random_state=42)
        if y_pred is None:
            raise ValueError("CV split too small")

        r_cv, Q2, rmse = cv_metrics(y, y_pred)

        print(f"r_cv   = {r_cv:.4f}" if np.isfinite(r_cv) else "r_cv   = NaN")
        print(f"Q2     = {Q2:.4f}" if np.isfinite(Q2) else "Q2     = NaN")
        print(f"RMSECV = {rmse:.4f}" if np.isfinite(rmse) else "RMSECV = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(
            X, pd.Series(y, index=df_xy.index),
            n_components=1,
            n_splits=5,
            random_state=42
        )

        w_tbl = full_tbl.join(cv_tbl, how="left")
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_cv": r_cv, "Q2": Q2, "RMSECV": rmse, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": f"FAIL: {type(e).__name__}"})


# 마지막 요약: CV Q2 기준 정렬(내림차순)
res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("Q2", ascending=False, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by CV Q2, desc)")
print(sep)
print(res_ok[["y_col", "n", "r_cv", "Q2", "RMSECV"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/10] TSubset = CD4trm
n = 13
r_cv   = 0.0558
Q2     = -0.0615
RMSECV = 17.3531
------------------------------------------------------------
PLSR Weights / Coefs
         x_weight_c1    coef  cv_w_mean  cv_w_std
Albumin       0.8377 -5.1418     0.8445    0.1007
Protein       0.5462 -2.5223     0.4791    0.2476
------------------------------------------------------------
__________________________________________________________________________________________
[2/10] TSubset = CD8trm
n = 13
r_cv   = 0.1012
Q2     = -0.0200
RMSECV = 17.5254
------------------------------------------------------------
PLSR Weights / Coefs
         x_weight_c1    coef  cv_w_mean  cv_w_std
Albumin       0.8414 -4.7462     0.8323    0.0828
Protein       0.5404 -2.2931     0.4042    0.4161
------------------------------------------------------------
______________________________________________________________________

In [ ]:
# df2 + 간 - 안나와야댐
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
from scipy.stats import pearsonr

sep = "_" * 90

# =========================
# df2 간(liver)
# =========================
X_cols = ['AST', 'ALT', 'ALP', 'Bilirubin']

y_candidates = [ 'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환(원래 순서대로)"""
    n = len(y)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def cv_metrics(y_true, y_pred):
    """r_cv, Q2, RMSECV 계산"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if (np.std(y_true) < 1e-12) or (np.std(y_pred) < 1e-12):
        r = np.nan
    else:
        r, _ = pearsonr(y_true, y_pred)

    sse = np.sum((y_true - y_pred) ** 2)
    sst = np.sum((y_true - np.mean(y_true)) ** 2)
    Q2 = np.nan if sst < 1e-12 else (1.0 - sse / sst)

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return r, Q2, rmse

def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    ※ PLS weight는 부호가 뒤집혀도 동등 → fold weight는 full-fit weight에 맞춰 부호 정렬
    """
    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()  # component1

        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행 (CV 기준)
# =========================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df2.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col].astype(float).to_numpy()

    try:
        y_pred = cv_pred_plsr(X, y, n_components=1, n_splits=5, random_state=42)
        if y_pred is None:
            raise ValueError("CV split too small")

        r_cv, Q2, rmse = cv_metrics(y, y_pred)

        print(f"r_cv   = {r_cv:.4f}" if np.isfinite(r_cv) else "r_cv   = NaN")
        print(f"Q2     = {Q2:.4f}" if np.isfinite(Q2) else "Q2     = NaN")
        print(f"RMSECV = {rmse:.4f}" if np.isfinite(rmse) else "RMSECV = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(
            X, pd.Series(y, index=df_xy.index),
            n_components=1,
            n_splits=5,
            random_state=42
        )

        w_tbl = full_tbl.join(cv_tbl, how="left")
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_cv": r_cv, "Q2": Q2, "RMSECV": rmse, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": f"FAIL: {type(e).__name__}"})


# 마지막 요약: CV Q2 기준 정렬(내림차순)
res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("Q2", ascending=False, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by CV Q2, desc)")
print(sep)
print(res_ok[["y_col", "n", "r_cv", "Q2", "RMSECV"]].to_string(index=False))
print(sep)


In [15]:
# df1 + 간 1
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
from scipy.stats import pearsonr

sep = "_" * 90

# =========================
# df2 간(liver)
# =========================
X_cols = ['BUN', 'Cr']

y_candidates = [ 'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

def cv_pred_plsr(X: pd.DataFrame, y: np.ndarray, n_components=1, n_splits=5, random_state=42):
    """PLSR로 K-fold CV 예측값 반환(원래 순서대로)"""
    n = len(y)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        y_pred[te_idx] = model.predict(X.iloc[te_idx]).reshape(-1)

    return y_pred

def cv_metrics(y_true, y_pred):
    """r_cv, Q2, RMSECV 계산"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if (np.std(y_true) < 1e-12) or (np.std(y_pred) < 1e-12):
        r = np.nan
    else:
        r, _ = pearsonr(y_true, y_pred)

    sse = np.sum((y_true - y_pred) ** 2)
    sst = np.sum((y_true - np.mean(y_true)) ** 2)
    Q2 = np.nan if sst < 1e-12 else (1.0 - sse / sst)

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return r, Q2, rmse

def plsr_weight_tables(X: pd.DataFrame, y: pd.Series, n_components=1, n_splits=5, random_state=42):
    """
    1) 전체 데이터로 fit한 x_weights_, coef_ 테이블
    2) CV fold별 x_weights_ 요약(mean±std) 테이블
    ※ PLS weight는 부호가 뒤집혀도 동등 → fold weight는 full-fit weight에 맞춰 부호 정렬
    """
    y_arr = y.astype(float).to_numpy().reshape(-1, 1)

    # (1) Full fit
    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X, y_arr)

    w_full = full.x_weights_
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X.columns
    )
    full_tbl["coef"] = coef

    # (2) CV weights summary
    n = len(X)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X.columns, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(X):
        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X.iloc[tr_idx], y_arr[tr_idx])

        w = m.x_weights_[:, 0].copy()  # component1

        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X.columns
    )

    return full_tbl, cv_tbl


# =========================
# 실행 (CV 기준)
# =========================
rows = []
###############################################
for i, y_col in enumerate(y_candidates, start=1):
    df_xy = df1.dropna(subset=X_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": "SKIP"})
        continue

    X = df_xy[X_cols]
    y = df_xy[y_col].astype(float).to_numpy()

    try:
        y_pred = cv_pred_plsr(X, y, n_components=1, n_splits=5, random_state=42)
        if y_pred is None:
            raise ValueError("CV split too small")

        r_cv, Q2, rmse = cv_metrics(y, y_pred)

        print(f"r_cv   = {r_cv:.4f}" if np.isfinite(r_cv) else "r_cv   = NaN")
        print(f"Q2     = {Q2:.4f}" if np.isfinite(Q2) else "Q2     = NaN")
        print(f"RMSECV = {rmse:.4f}" if np.isfinite(rmse) else "RMSECV = NaN")

        full_tbl, cv_tbl = plsr_weight_tables(
            X, pd.Series(y, index=df_xy.index),
            n_components=1,
            n_splits=5,
            random_state=42
        )

        w_tbl = full_tbl.join(cv_tbl, how="left")
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_cv": r_cv, "Q2": Q2, "RMSECV": rmse, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": f"FAIL: {type(e).__name__}"})


# 마지막 요약: CV Q2 기준 정렬(내림차순)
res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("Q2", ascending=False, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by CV Q2, desc)")
print(sep)
print(res_ok[["y_col", "n", "r_cv", "Q2", "RMSECV"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/10] TSubset = CD4trm
n = 13
r_cv   = -0.0588
Q2     = -0.3979
RMSECV = 19.9140
------------------------------------------------------------
PLSR Weights / Coefs
     x_weight_c1     coef  cv_w_mean  cv_w_std
Cr         0.895 -30.6042     0.8487    0.1818
BUN       -0.446   1.3016    -0.4460    0.2605
------------------------------------------------------------
__________________________________________________________________________________________
[2/10] TSubset = CD8trm
n = 13
r_cv   = -0.1508
Q2     = -0.4136
RMSECV = 20.6309
------------------------------------------------------------
PLSR Weights / Coefs
     x_weight_c1     coef  cv_w_mean  cv_w_std
Cr        0.9579 -31.2569     0.9554    0.0223
BUN      -0.2870   0.7991    -0.2868    0.0755
------------------------------------------------------------
______________________________________________________________________________________

bmi 통제 + 간신장 나누기 + pls

In [18]:
# ======================================================
# PLSR (BUN/Cr) with covariate control (BMI, Sex_01)
# - CV prediction metrics: r_cv, Q2, RMSECV
# - Full-fit weights/coef + CV-fold weights summary (mean±std)
# ======================================================

import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr

sep = "_" * 90

# =========================
# 설정
# =========================
# 주효과(설명변수): 간/신장 지표 등
X_cols = ['BUN', 'Cr']

# 통제변수(공변량)
C_cols = ['BMI']   # Sex_01은 0/1 인코딩되어 있다고 가정

# 반응변수 후보
y_candidates = [
    'CD4trm', 'CD8trm', 'MAITtrm', 'GD1Ttrm', 'GD2Ttrm',
    'CD4grzk', 'CD8grzk', 'MAITgrzK', 'GD1TgrzK', 'GD2TgrzK',
]

# ======================================================
# 1) 유틸: CV metric
# ======================================================
def cv_metrics(y_true, y_pred):
    """r_cv, Q2, RMSECV 계산"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if (np.std(y_true) < 1e-12) or (np.std(y_pred) < 1e-12):
        r = np.nan
    else:
        r, _ = pearsonr(y_true, y_pred)

    sse = np.sum((y_true - y_pred) ** 2)
    sst = np.sum((y_true - np.mean(y_true)) ** 2)
    Q2 = np.nan if sst < 1e-12 else (1.0 - sse / sst)

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return r, Q2, rmse

# ======================================================
# 2) 잔차화(residualization) 함수
#    - train에서 공변량 회귀 fit
#    - train/test에 동일 모델로 적용
# ======================================================
def residualize_X_train_test(X_tr: pd.DataFrame, X_te: pd.DataFrame,
                             C_tr: pd.DataFrame, C_te: pd.DataFrame):
    """
    X의 각 컬럼을 공변량 C로 선형회귀 후, 잔차 반환
    """
    X_tr_res = X_tr.copy()
    X_te_res = X_te.copy()

    # 안정성: numpy로 변환
    Ctr = np.asarray(C_tr, dtype=float)
    Cte = np.asarray(C_te, dtype=float)

    for col in X_tr.columns:
        yx_tr = np.asarray(X_tr[col], dtype=float)
        lr = LinearRegression()
        lr.fit(Ctr, yx_tr)

        X_tr_res[col] = yx_tr - lr.predict(Ctr)
        X_te_res[col] = np.asarray(X_te[col], dtype=float) - lr.predict(Cte)

    return X_tr_res, X_te_res

def residualize_y_train_test(y_tr: np.ndarray, y_te: np.ndarray,
                             C_tr: pd.DataFrame, C_te: pd.DataFrame):
    """
    y를 공변량 C로 선형회귀 후, 잔차 반환
    """
    Ctr = np.asarray(C_tr, dtype=float)
    Cte = np.asarray(C_te, dtype=float)

    lr = LinearRegression()
    lr.fit(Ctr, np.asarray(y_tr, dtype=float))

    y_tr_res = np.asarray(y_tr, dtype=float) - lr.predict(Ctr)
    y_te_res = np.asarray(y_te, dtype=float) - lr.predict(Cte)
    return y_tr_res, y_te_res

# ======================================================
# 3) CV 예측 (통제 포함, 누수 방지)
# ======================================================
def cv_pred_plsr_controlled(df_xy: pd.DataFrame,
                            X_cols, y_col, C_cols,
                            n_components=1, n_splits=5, random_state=42):
    """
    K-fold CV에서:
      (1) train에서 공변량 통제(잔차화) 모델 학습
      (2) train/test를 잔차화
      (3) 잔차화된 X로 PLSR 학습 → test 예측
    반환: 원래 인덱스 순서의 y_pred (np.ndarray)
    """
    n = len(df_xy)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        return None

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    y_all = df_xy[y_col].astype(float).to_numpy()
    y_pred = np.empty(n, dtype=float)

    for tr_idx, te_idx in cv.split(df_xy):
        X_tr = df_xy.iloc[tr_idx][X_cols]
        X_te = df_xy.iloc[te_idx][X_cols]
        C_tr = df_xy.iloc[tr_idx][C_cols]
        C_te = df_xy.iloc[te_idx][C_cols]

        y_tr = y_all[tr_idx]
        y_te = y_all[te_idx]

        # ✅ fold 내부 통제(잔차화)
        X_tr_res, X_te_res = residualize_X_train_test(X_tr, X_te, C_tr, C_te)
        y_tr_res, _ = residualize_y_train_test(y_tr, y_te, C_tr, C_te)

        # ✅ PLSR
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X_tr_res, y_tr_res.reshape(-1, 1))
        y_pred[te_idx] = model.predict(X_te_res).reshape(-1)

    return y_pred

# ======================================================
# 4) weight/coef 테이블 (통제 포함)
#    - Full-fit: 전체 데이터로 공변량 회귀 fit → 잔차화 → PLSR fit
#    - CV fold weights: 각 fold에서 train 기반 잔차화 후 weights 추출
#      (PLS weight 부호는 뒤집혀도 동등 → full-fit weight에 맞춰 sign align)
# ======================================================
def residualize_X_full(X: pd.DataFrame, C: pd.DataFrame):
    """전체 데이터 기준으로 X 잔차화"""
    X_res = X.copy()
    Cmat = np.asarray(C, dtype=float)

    for col in X.columns:
        lr = LinearRegression().fit(Cmat, np.asarray(X[col], dtype=float))
        X_res[col] = np.asarray(X[col], dtype=float) - lr.predict(Cmat)
    return X_res

def residualize_y_full(y: pd.Series, C: pd.DataFrame):
    """전체 데이터 기준으로 y 잔차화"""
    Cmat = np.asarray(C, dtype=float)
    yv = np.asarray(y.astype(float), dtype=float)
    lr = LinearRegression().fit(Cmat, yv)
    return yv - lr.predict(Cmat)

def plsr_weight_tables_controlled(df_xy: pd.DataFrame,
                                  X_cols, y_col, C_cols,
                                  n_components=1, n_splits=5, random_state=42):
    """
    반환:
      full_tbl: x_weight_c1.., coef (전체 잔차화 후 full fit)
      cv_tbl  : cv_w_mean, cv_w_std (fold별 x_weights_ 요약, sign align)
    """
    X = df_xy[X_cols]
    C = df_xy[C_cols]
    y = df_xy[y_col].astype(float)

    # (1) Full fit (전체 기준 잔차화)
    X_res_full = residualize_X_full(X, C)
    y_res_full = residualize_y_full(y, C).reshape(-1, 1)

    full = PLSRegression(n_components=n_components, scale=True)
    full.fit(X_res_full, y_res_full)

    w_full = full.x_weights_  # (p, n_components)
    coef = full.coef_.reshape(-1)

    full_tbl = pd.DataFrame(
        {f"x_weight_c{k+1}": w_full[:, k] for k in range(n_components)},
        index=X_cols
    )
    full_tbl["coef"] = coef

    # (2) CV fold weights summary
    n = len(df_xy)
    n_splits = min(n_splits, n)
    if n_splits < 2:
        cv_tbl = pd.DataFrame(index=X_cols, data={"cv_w_mean": np.nan, "cv_w_std": np.nan})
        return full_tbl, cv_tbl

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_ws = []

    # sign align reference: full-fit component1 weight
    ref = w_full[:, 0].copy()

    for tr_idx, te_idx in cv.split(df_xy):
        X_tr = df_xy.iloc[tr_idx][X_cols]
        C_tr = df_xy.iloc[tr_idx][C_cols]
        y_tr = df_xy.iloc[tr_idx][y_col].astype(float)

        # train 기준 잔차화
        X_tr_res = residualize_X_full(X_tr, C_tr)   # train-only fit이지만 함수는 내부에서 train만 사용
        y_tr_res = residualize_y_full(y_tr, C_tr).reshape(-1, 1)

        m = PLSRegression(n_components=n_components, scale=True)
        m.fit(X_tr_res, y_tr_res)

        w = m.x_weights_[:, 0].copy()

        # sign align
        if np.dot(w, ref) < 0:
            w *= -1
        fold_ws.append(w)

    fold_ws = np.vstack(fold_ws)
    cv_tbl = pd.DataFrame(
        {
            "cv_w_mean": fold_ws.mean(axis=0),
            "cv_w_std": fold_ws.std(axis=0, ddof=1) if fold_ws.shape[0] > 1 else np.zeros(fold_ws.shape[1]),
        },
        index=X_cols
    )

    return full_tbl, cv_tbl

# ======================================================
# 5) 실행 루프
# ======================================================
rows = []

for i, y_col in enumerate(y_candidates, start=1):
    # df2가 이미 존재한다고 가정
    df_xy = df1.dropna(subset=X_cols + C_cols + [y_col]).copy()
    n = len(df_xy)

    print(sep)
    print(f"[{i}/{len(y_candidates)}] TSubset = {y_col}")
    print(f"n = {n}")

    if n < 6:
        print("→ SKIP (n too small)")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": "SKIP"})
        continue

    try:
        # ✅ 통제 포함 CV 예측
        y_true = df_xy[y_col].astype(float).to_numpy()
        y_pred = cv_pred_plsr_controlled(
            df_xy=df_xy,
            X_cols=X_cols,
            y_col=y_col,
            C_cols=C_cols,
            n_components=1,
            n_splits=5,
            random_state=42
        )
        if y_pred is None:
            raise ValueError("CV split too small")

        r_cv, Q2, rmse = cv_metrics(y_true, y_pred)

        print(f"r_cv   = {r_cv:.4f}" if np.isfinite(r_cv) else "r_cv   = NaN")
        print(f"Q2     = {Q2:.4f}" if np.isfinite(Q2) else "Q2     = NaN")
        print(f"RMSECV = {rmse:.4f}" if np.isfinite(rmse) else "RMSECV = NaN")

        # ✅ 통제 포함 weight/coef
        full_tbl, cv_tbl = plsr_weight_tables_controlled(
            df_xy=df_xy,
            X_cols=X_cols,
            y_col=y_col,
            C_cols=C_cols,
            n_components=1,
            n_splits=5,
            random_state=42
        )

        w_tbl = full_tbl.join(cv_tbl, how="left")
        w_tbl = w_tbl.reindex(w_tbl["coef"].abs().sort_values(ascending=False).index)

        print("-" * 60)
        print("PLSR Weights / Coefs (controlled for BMI, Sex_01)")
        print(w_tbl.round(4).to_string())
        print("-" * 60)

        rows.append({"y_col": y_col, "n": n, "r_cv": r_cv, "Q2": Q2, "RMSECV": rmse, "status": "OK"})

    except Exception as e:
        print(f"→ FAIL: {type(e).__name__}: {str(e)[:120]}")
        rows.append({"y_col": y_col, "n": n, "r_cv": np.nan, "Q2": np.nan, "RMSECV": np.nan, "status": f"FAIL: {type(e).__name__}"})


# ======================================================
# 6) 마지막 요약: CV Q2 기준 정렬(내림차순)
# ======================================================
res = pd.DataFrame(rows)
res_ok = res.query("status == 'OK'").sort_values("Q2", ascending=False, na_position="last")

print(sep)
print("FINAL SUMMARY (sorted by CV Q2, desc)  [controlled: BMI, Sex_01]")
print(sep)
if len(res_ok) == 0:
    print("No OK results.")
else:
    print(res_ok[["y_col", "n", "r_cv", "Q2", "RMSECV"]].to_string(index=False))
print(sep)


__________________________________________________________________________________________
[1/10] TSubset = CD4trm
n = 13
r_cv   = -0.1027
Q2     = -13.1079
RMSECV = 63.2627
------------------------------------------------------------
PLSR Weights / Coefs (controlled for BMI, Sex_01)
     x_weight_c1     coef  cv_w_mean  cv_w_std
Cr        0.8966 -30.8005     0.7473    0.3962
BUN      -0.4429   1.3604    -0.3652    0.4777
------------------------------------------------------------
__________________________________________________________________________________________
[2/10] TSubset = CD8trm
n = 13
r_cv   = -0.2830
Q2     = -9.7546
RMSECV = 56.9056
------------------------------------------------------------
PLSR Weights / Coefs (controlled for BMI, Sex_01)
     x_weight_c1     coef  cv_w_mean  cv_w_std
Cr        0.9614 -31.3096     0.7778    0.2752
BUN      -0.2751   0.8010    -0.4873    0.3483
------------------------------------------------------------
___________________________